# 🧠 Python Internals & Concurrency — How Python Really Runs Your Code

> **What you'll learn:** what variables really are, how Python frees memory, why mutable objects cause surprise bugs, how closures and dictionaries work under the hood, why the GIL exists, and when to use threads, processes, or `asyncio` — with every claim *measured* on your own machine.

| | |
|---|---|
| **Difficulty** | 🟢 Beginner → 🔴 Advanced (ramps up section by section) |
| **Time** | ~4 hours to read and run, +2 hours for exercises and the project |
| **Prerequisites** | [Python Basics](01_Python_Basics.ipynb) · [Python Built-ins](02_Python_Builtins.ipynb) · [OOP in Python](03_OOP_in_Python.ipynb) |
| **Tested with** | Python 3.12 · httpx 0.28 (everything else is the standard library) |
| **Interview relevance** | ⭐⭐⭐ High — "`is` vs `==`", mutable default arguments, late-binding closures, "what is the GIL?", threads vs processes vs asyncio, "call an LLM API 1,000 times with a rate limit", implement an LRU cache |

## 🤔 What Is Python Internals & Concurrency?

**Python internals** means *what actually happens* when Python runs your code: how values are stored, when memory is freed, how a dictionary finds a key in one step, and what the interpreter does line by line.

**Concurrency** means making progress on several tasks during the same period of time. Think of a restaurant:

- **Sequential:** one waiter takes an order, waits in the kitchen until the food is ready, serves it, and only then goes to the next table.
- **Threads:** several waiters share one kitchen. Great when most of the time is spent *waiting* — but only one of them may use the single chef's knife at a time (that knife is Python's **GIL**).
- **Processes:** several complete restaurants, each with its own kitchen and knife. Real parallel cooking, but opening a restaurant is expensive.
- **asyncio:** one very organised waiter who takes an order, and *while the kitchen cooks*, goes to the next table instead of standing still.

```
Sequential   [task A ██████][task B ██████][task C ██████]      → slow when tasks mostly wait
Threads      [A ██████]                                          → good for waiting (I/O)
             [B ██████]      only one thread runs Python code at any instant
             [C ██████]
Processes    [A ██████] core 1                                   → true parallel CPU work
             [B ██████] core 2
asyncio      [A ▒▒][B ▒▒][C ▒▒][A ▒▒][B ▒▒]...  one thread, switches at every `await`
```

## 🎯 Why It Matters

- **Bugs you will actually hit:** a config dict that changes "by itself", a list of callbacks that all use the last value, a default argument that remembers old calls, a counter that loses updates. All of them come from the internals in this notebook.
- **AI engineering is full of waiting and crunching.** Calling an LLM or embedding API thousands of times is *waiting* (use `asyncio` or threads with a rate limit). Tokenizing or augmenting data in pure Python is *crunching* (use processes — that's what PyTorch `DataLoader(num_workers=...)` does).
- **Performance intuition:** knowing that `x in list` is O(n) but `x in set` is O(1) turns a 10-minute job into a 1-second job.
- **In interviews** these are standard screening questions for Python-heavy AI/ML roles: *"What is the GIL?"*, *"Threads or processes for this workload?"*, *"Why does this closure print 2, 2, 2?"*, *"Implement an LRU cache"*, *"Send 10,000 prompts to an API without getting rate-limited."*

## ✅ By the End You Can

- [ ] Explain names vs objects, `is` vs `==`, reference counting, and garbage collection
- [ ] Predict aliasing bugs and fix them with the right kind of copy
- [ ] Explain closures, LEGB scope, and the late-binding lambda trap
- [ ] Explain how dicts and sets use hashing, and choose data structures by their measured cost
- [ ] Explain what the GIL protects and choose threads, processes, or `asyncio` for a workload — with numbers
- [ ] Build a thread pool, an async rate limiter, and an LRU cache from scratch

## 📋 Table of Contents

1. [Everything Is an Object](#1.-Everything-Is-an-Object:-id,-type,-and-is-🟢)
2. [Names, References, and Reference Counting](#2.-Names,-References,-and-Reference-Counting-🟢)
3. [Garbage Collection and Reference Cycles](#3.-Garbage-Collection-and-Reference-Cycles-🟡)
4. [Mutability Revisited: Aliasing and Copies](#4.-Mutability-Revisited:-Aliasing,-copy,-and-deepcopy-🟢)
5. [Scope: LEGB, global, and nonlocal](#5.-Scope:-LEGB,-global,-and-nonlocal-🟢)
6. [Closures and the Late-Binding Trap](#6.-Closures-and-the-Late-Binding-Trap-🟡)
7. [How Dicts and Sets Work: Hashing](#7.-How-Dicts-and-Sets-Work:-Hashing-🟡)
8. [Time Complexity of Built-ins, Measured](#8.-Time-Complexity-of-Built-ins,-Measured-🟡)
9. [Generators vs Lists: Memory, Measured](#9.-Generators-vs-Lists:-Memory,-Measured-🟢)
10. [Peeking at Bytecode with dis](#10.-Peeking-at-Bytecode-with-dis-🟡)
11. [Threads and the GIL](#11.-Threads-and-the-GIL-🟡)
12. [Processes and concurrent.futures](#12.-Processes-and-concurrent.futures-🟡)
13. [asyncio: async, await, gather, Semaphores, Timeouts](#13.-asyncio:-async,-await,-gather,-Semaphores,-Timeouts-🔴)
14. [Concurrency in AI Work](#14.-Concurrency-in-AI-Work-🔴)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-Concurrent-Dataset-Downloader) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

Run the cell below first. Everything except `httpx` (a modern HTTP client, used for real downloads) ships with Python. If `httpx` is missing, uncomment the `%pip` line, run it once, then restart the kernel.

Two helpers are defined here:
- `check()` gives instant feedback on exercises: **✅** correct, **⏳** not attempted yet, **❌** wrong (with a hint).
- `Timer` measures how long a block of code takes: `with Timer() as t: ...` then read `t.seconds`.

> Timings in this notebook are **real measurements on your machine**, so your numbers will differ from anyone else's. The printed conclusions are computed from *your* numbers.

In [1]:
# %pip install -q "httpx>=0.27"

import asyncio
import copy
import dis
import gc
import hashlib
import importlib
import itertools
import multiprocessing as mp
import os
import queue
import random
import statistics
import subprocess
import sys
import textwrap
import threading
import time
import timeit
import tracemalloc
import weakref
from collections import OrderedDict, deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from functools import lru_cache, partial
from pathlib import Path

import httpx

print(f"Python {sys.version.split()[0]} | httpx {httpx.__version__} | CPU cores: {os.cpu_count()}")
print(f"multiprocessing start method on this OS: {mp.get_start_method()}")
gil_probe = getattr(sys, "_is_gil_enabled", None)          # this function only exists on Python 3.13+
print("GIL enabled:", gil_probe() if gil_probe else "yes (Python < 3.13 always has a GIL)")

random.seed(42)
OUTPUT_DIR = Path("_outputs")                              # files we write go here (ignored by git)
OUTPUT_DIR.mkdir(exist_ok=True)


class Timer:
    """Usage: `with Timer() as t: ...` → `t.seconds` is the elapsed wall-clock time."""

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc_info):
        self.seconds = time.perf_counter() - self.start


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    if isinstance(expected, float) and isinstance(got, (int, float)):
        ok = abs(got - expected) <= 1e-9 * max(1.0, abs(expected))
    else:
        ok = got == expected
    assert ok, f"❌ {name}: not quite (got {got!r}). {hint}"
    print(f"✅ {name}: correct!")

Python 3.12.11 | httpx 0.28.1 | CPU cores: 15
multiprocessing start method on this OS: spawn
GIL enabled: yes (Python < 3.13 always has a GIL)


## 1. Everything Is an Object: id, type, and is 🟢

In Python **every value is an object**: numbers, strings, lists, functions, classes, even modules. Every object has three things:

| Property | Question it answers | How to see it |
|---|---|---|
| **identity** | *Which* object is this? (its address in memory) | `id(obj)` |
| **type** | What kind of object? What can it do? | `type(obj)` |
| **value** | What data does it hold? | printing it, `==` |

**The one rule:** `==` asks *"do these have equal values?"*; `is` asks *"are these the very same object?"* Two printed copies of the same book are `==`; only one physical book `is` itself.

In [2]:
scores = [90, 85]
same_list = scores           # a second NAME for the SAME object
copy_list = list(scores)     # a NEW object with equal contents

print("type:", type(scores).__name__, "| id:", id(scores))
print("same_list is scores :", same_list is scores)
print("copy_list is scores :", copy_list is scores)
print("copy_list == scores :", copy_list == scores)

# Functions, classes and modules are objects too: they have a type and can be stored in variables
for thing in (42, "hi", len, int, sys):
    print(f"{thing!r:<28} → type {type(thing).__name__}")

type: list | id: 4451230848
same_list is scores : True
copy_list is scores : False
copy_list == scores : True
42                           → type int
'hi'                         → type str
<built-in function len>      → type builtin_function_or_method
<class 'int'>                → type type
<module 'sys' (built-in)>    → type module


In [3]:
# CPython caches small integers (-5..256), so `is` can LOOK like it works for numbers — it doesn't
a, b = int("256"), int("256")          # int("...") builds the number at run time
c, d = int("257"), int("257")
print("256 is 256:", a is b, "| 257 is 257:", c is d, "| 257 == 257:", c == d)

result = None
print("Testing for None → use `is`:", result is None)

256 is 256: True | 257 is 257: False | 257 == 257: True
Testing for None → use `is`: True


### ✍️ Your Turn

**Predict first, then check.** Replace `None` with a tuple of three booleans: `(x is y, x is z, x == z)`.

In [4]:
x = [1, 2, 3]
y = x
z = x[:]
answers = None  # TODO: e.g. (True, True, True)
check("identity_vs_equality", answers, (True, False, True), hint="Slicing a list with [:] builds a brand-new list object.")

⏳ identity_vs_equality: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
x = [1, 2, 3]
y = x
z = x[:]
answers = (True, False, True)       # the same as (x is y, x is z, x == z)
check("identity_vs_equality", answers, (True, False, True))
```

`y = x` copies nothing — it adds a second name. `x[:]` creates a new list with equal contents.
</details>

> 💡 **Interview angle:** "`is` vs `==`?" — `is` compares identity (same object), `==` compares values (calls `__eq__`). Use `is` only for singletons like `None`; `257 is 257` style checks depend on caching details and are a bug.

## 2. Names, References, and Reference Counting 🟢

A variable is **not a box that holds a value**. It is a **name tag** tied to an object.

```
  data  ─┐
         ├──►  [1, 2, 3]      one list object, two name tags
  alias ─┘
```

- **Assignment never copies.** `alias = data` ties a second tag to the same object.
- CPython keeps a **reference count** on every object: how many tags, list slots, dict values, etc. point to it.
- When the count reaches **0**, the object is freed **immediately**.

In [5]:
data = [1, 2, 3]
print("refcount of a fresh list  :", sys.getrefcount(data))   # always +1: the argument to getrefcount itself
alias = data
print("after alias = data        :", sys.getrefcount(data))
holder = {"key": data}
print("after storing it in a dict:", sys.getrefcount(data))
del alias
print("after del alias           :", sys.getrefcount(data))
print("refcount of None          :", sys.getrefcount(None), "← 'immortal' object since Python 3.12 (PEP 683)")

refcount of a fresh list  : 2
after alias = data        : 3
after storing it in a dict: 4
after del alias           : 3
refcount of None          : 4294967295 ← 'immortal' object since Python 3.12 (PEP 683)


In [6]:
class Model:
    def __init__(self, name):
        self.name = name


m = Model("bert")
watcher = weakref.ref(m)        # a WEAK reference lets us watch an object without keeping it alive
print("alive before del:", watcher() is not None)
del m                           # the last strong reference is gone → refcount 0 → freed right now
print("alive after del :", watcher() is not None)

alive before del: True
alive after del : False


**Passing arguments works the same way.** The function's parameter is just another name tag on the caller's object. That's why Python is called **"pass by object reference"** (or "call by sharing"): mutating the object is visible to the caller, rebinding the name is not.

In [7]:
def add_item(items, item):
    items.append(item)          # MUTATES the caller's object → caller sees it
    items = ["brand", "new"]    # REBINDS the local name only → caller does not see it
    return items


shopping = ["milk"]
returned = add_item(shopping, "eggs")
print("caller's list:", shopping, "| returned:", returned)

caller's list: ['milk', 'eggs'] | returned: ['brand', 'new']


> 💡 **Interview angle:** "Is Python pass-by-value or pass-by-reference?" — neither: *pass by object reference*. The function gets a new name for the same object; in-place changes are shared, reassignment is not.

## 3. Garbage Collection and Reference Cycles 🟡

Reference counting has one blind spot: **cycles**. If `a` points to `b` and `b` points back to `a`, each keeps the other's count above 0 even when nothing else can reach them.

```
  (no names left)     a ──► b
                      ▲     │
                      └─────┘     refcounts never reach 0
```

CPython runs a separate **cyclic garbage collector** (`gc` module) now and then to find such unreachable groups. Objects are tracked in *generations*: most objects die young, so new objects are checked often and old survivors rarely.

In [8]:
class Node:
    def __init__(self, name):
        self.name, self.partner = name, None


gc.disable()                                    # pause the automatic collector so we can watch
a, b = Node("a"), Node("b")
a.partner, b.partner = b, a                     # a → b → a : a reference cycle
watch_a = weakref.ref(a)
del a, b                                        # no names left, but they still point at each other
print("after del, cycle still alive:", watch_a() is not None)

found = gc.collect()                            # run the cycle detector by hand
print(f"gc.collect() found {found} unreachable objects | alive now: {watch_a() is not None}")
gc.enable()
print("automatic collection thresholds:", gc.get_threshold())

after del, cycle still alive: True
gc.collect() found 2 unreachable objects | alive now: False
automatic collection thresholds: (700, 10, 10)


**Why an AI engineer cares:** in PyTorch, `losses.append(loss)` keeps the *whole computation graph* of every step alive (the tensor references it), and GPU memory climbs until it crashes. `losses.append(loss.item())` stores a plain float instead. Same idea: *memory is freed when the last reference goes away* — so find the reference.

> 💡 **Interview angle:** "Does `del x` free memory?" — `del` removes a *name*. The object is freed when its reference count hits 0; objects stuck in cycles wait for the cyclic GC. Even then CPython may keep the memory for reuse instead of returning it to the OS.

## 4. Mutability Revisited: Aliasing, copy, and deepcopy 🟢

| Immutable (can't change after creation) | Mutable (can change in place) |
|---|---|
| `int`, `float`, `bool`, `str`, `tuple`, `frozenset`, `bytes` | `list`, `dict`, `set`, `bytearray`, most class instances |

**Aliasing** = two names for one mutable object, so a change through one name shows up through the other. Two kinds of copies:

- **Shallow copy** (`copy.copy`, `list(x)`, `x[:]`, `dict(x)`): a new *outer* container, but the items inside are shared.
- **Deep copy** (`copy.deepcopy`): copies everything, all the way down.

In [9]:
row = [0] * 3
grid_bad = [row] * 3                      # ❌ a list holding the SAME row object three times
grid_bad[0][0] = 1
print("[[0]*3]*3 after grid[0][0] = 1 :", grid_bad, "| rows are one object:", grid_bad[0] is grid_bad[2])

grid_good = [[0] * 3 for _ in range(3)]  # ✅ the comprehension builds a new row each time
grid_good[0][0] = 1
print("comprehension after the same op:", grid_good)

[[0]*3]*3 after grid[0][0] = 1 : [[1, 0, 0], [1, 0, 0], [1, 0, 0]] | rows are one object: True
comprehension after the same op: [[1, 0, 0], [0, 0, 0], [0, 0, 0]]


In [10]:
base_config = {"model": "resnet18", "lr": 1e-3, "augment": ["flip", "crop"]}

shallow = copy.copy(base_config)
deep = copy.deepcopy(base_config)
shallow["lr"] = 0.1                       # top-level value: the copy has its own slot
shallow["augment"].append("rotate")       # nested list: SHARED between base_config and shallow

print("original :", base_config)
print("deep copy:", deep)
print("nested list shared?  shallow:", shallow["augment"] is base_config["augment"],
      "| deep:", deep["augment"] is base_config["augment"])

original : {'model': 'resnet18', 'lr': 0.001, 'augment': ['flip', 'crop', 'rotate']}
deep copy: {'model': 'resnet18', 'lr': 0.001, 'augment': ['flip', 'crop']}
nested list shared?  shallow: True | deep: False


In [11]:
x = [1, 2, 3]
y = x
x = x + [4]          # builds a NEW list, then points x at it
print("after x = x + [4] → y is", y)

x = [1, 2, 3]
y = x
x += [4]             # extends the SAME list in place (list.__iadd__)
print("after x += [4]    → y is", y)

after x = x + [4] → y is [1, 2, 3]
after x += [4]    → y is [1, 2, 3, 4]


### ✍️ Your Turn

You are launching a hyper-parameter sweep. Build `configs`: a list of **3 fully independent copies** of `base`, so that editing one config's nested `layers` list leaves `base` and the other configs untouched.

In [12]:
base = {"lr": 0.01, "layers": [64, 32]}
configs = None  # TODO: a list of 3 independent copies of base
if configs is not None:
    configs[0]["layers"].append(16)
check("independent_configs", None if configs is None else [base, configs[1]],
      [{"lr": 0.01, "layers": [64, 32]}, {"lr": 0.01, "layers": [64, 32]}],
      hint="[base] * 3 and [dict(base) ...] both share the nested list — which copy goes all the way down?")

⏳ independent_configs: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
base = {"lr": 0.01, "layers": [64, 32]}
configs = [copy.deepcopy(base) for _ in range(3)]
configs[0]["layers"].append(16)
check("independent_configs", [base, configs[1]],
      [{"lr": 0.01, "layers": [64, 32]}, {"lr": 0.01, "layers": [64, 32]}])
```

`[base] * 3` gives three references to one dict; `[dict(base) for _ in range(3)]` gives three dicts that still share one `layers` list. Only `deepcopy` separates the nested list.
</details>

> 💡 **Interview angle:** "Shallow vs deep copy?" — shallow copies the container and shares the contents; deep copies recursively. Mention the `[[0]*n]*n` grid bug and that `x = x + y` rebinds while `x += y` mutates lists in place.

## 5. Scope: LEGB, global, and nonlocal 🟢

When Python meets a name, it searches four places **in this order** — the **LEGB** rule:

```
 B  Built-in   len, print, range ...                      ┐
 G  Global     names defined at the top of the module     │ searched from
 E  Enclosing  names in the outer function (if nested)    │ the inside out
 L  Local      names assigned inside the current function ┘
```

**The surprising rule:** if a function *assigns* to a name anywhere in its body, that name is **local for the whole function** — even on lines before the assignment. `global name` and `nonlocal name` tell Python to use the outer variable instead.

In [13]:
model_name = "global-model"                      # G

def outer():
    model_name = "enclosing-model"               # E (for inner)

    def inner():
        return f"inner sees {model_name!r}"      # not local → found in Enclosing

    return inner()

print(outer())
print("global is unchanged:", model_name)
import builtins
print("len comes from builtins:", len is builtins.len)

inner sees 'enclosing-model'
global is unchanged: global-model
len comes from builtins: True


In [14]:
counter = 0

def increment_wrong():
    counter += 1          # assignment makes `counter` LOCAL, but it has no local value yet

try:
    increment_wrong()
except UnboundLocalError as err:
    print("UnboundLocalError:", err)

def increment_global():
    global counter        # use the module-level name
    counter += 1

increment_global()
print("counter after increment_global():", counter)

UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value
counter after increment_global(): 1


In [15]:
def make_counter():
    count = 0             # lives in the enclosing scope of `step`

    def step():
        nonlocal count    # rebind the ENCLOSING variable, not a new local one
        count += 1
        return count

    return step


counter_a, counter_b = make_counter(), make_counter()
print("counter_a:", counter_a(), counter_a(), counter_a(), "| counter_b (independent):", counter_b())

counter_a: 1 2 3 | counter_b (independent): 1


### ✍️ Your Turn

Write `make_running_mean()`: it returns a function that takes a number, adds it to the history, and returns the **mean of all numbers seen so far**. Keep the state in the enclosing scope (no globals, no classes).

In [16]:
def make_running_mean():
    # TODO: store a running total and count here, define an inner function that uses `nonlocal`, return it
    return None


running_mean = make_running_mean()
check("running_mean", None if running_mean is None else [running_mean(10), running_mean(20), running_mean(60)],
      [10.0, 15.0, 30.0], hint="total += value and count += 1 inside the inner function need `nonlocal total, count`.")

⏳ running_mean: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def make_running_mean():
    total, count = 0.0, 0

    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    return add


running_mean = make_running_mean()
check("running_mean", [running_mean(10), running_mean(20), running_mean(60)], [10.0, 15.0, 30.0])
```
</details>

> 💡 **Interview angle:** "Why does `x += 1` inside a function raise `UnboundLocalError`?" — any assignment makes the name local at *compile* time, so the read on the right-hand side finds an unassigned local. Fix with `global`/`nonlocal`, or better, pass values in and return them.

## 6. Closures and the Late-Binding Trap 🟡

A **closure** is a function that *remembers variables from the scope where it was created*, even after that scope has finished. Decorators, callbacks, and learning-rate schedule lambdas are all closures.

**The trap:** a closure remembers the **variable**, not the **value it had** when the function was created. The value is looked up *later*, when the function is called — this is called **late binding**.

In [17]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor         # `factor` is a free variable, captured from make_multiplier
    return multiply


triple = make_multiplier(3)
print("triple(5) =", triple(5))
print("free variables:", triple.__code__.co_freevars, "| stored in a cell:", triple.__closure__[0].cell_contents)

triple(5) = 15
free variables: ('factor',) | stored in a cell: 3


In [18]:
callbacks = [lambda: i for i in range(3)]            # ❌ all three share ONE variable i
print("❌ late binding :", [f() for f in callbacks])

callbacks = [lambda i=i: i for i in range(3)]        # ✅ a default argument is evaluated NOW
print("✅ default arg  :", [f() for f in callbacks])

def show(i):
    return i

callbacks = [partial(show, i) for i in range(3)]     # ✅ partial also stores the current value
print("✅ partial      :", [f() for f in callbacks])

❌ late binding : [2, 2, 2]
✅ default arg  : [0, 1, 2]
✅ partial      : [0, 1, 2]


In [19]:
def build_callbacks():
    fns = []
    for i in range(3):
        fns.append(lambda: i)
    return fns


fns = build_callbacks()
print("all closures share the same cell:", fns[0].__closure__[0] is fns[2].__closure__[0])
print("and that cell holds the loop's LAST value:", fns[0].__closure__[0].cell_contents)

all closures share the same cell: True
and that cell holds the loop's LAST value: 2


### ✍️ Your Turn

Build `transforms`, a list of functions where `transforms[k](x)` returns `x ** powers[k]` — a typical way to create polynomial features. Avoid the late-binding trap.

In [20]:
powers = [1, 2, 3]
transforms = None  # TODO: a list of 3 functions
check("transforms", None if transforms is None else [t(2) for t in transforms], [2, 4, 8],
      hint="[lambda x: x ** p for p in powers] gives [8, 8, 8]. Bind p now with a default argument.")

⏳ transforms: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
powers = [1, 2, 3]
transforms = [lambda x, p=p: x ** p for p in powers]
check("transforms", [t(2) for t in transforms], [2, 4, 8])
```

`[partial(pow, exp=p) for p in powers]` also works (`pow` accepts `exp` as a keyword argument).
</details>

> 💡 **Interview angle:** "Why does `[lambda: i for i in range(3)]` return `[2, 2, 2]`?" — closures capture variables, not values; `i` is read at call time, after the loop ended. Fix with `lambda i=i: i` or `functools.partial`.

## 7. How Dicts and Sets Work: Hashing 🟡

A dict is like a coat check with numbered hooks:

1. `hash(key)` turns the key into a big integer → that picks a **hook** (bucket).
2. Python goes straight to that hook — no searching through every coat. That's why lookup is **O(1) on average**.
3. Because different keys can land on the same hook (a **collision**), Python confirms with `==`.

Two rules make this work:
- **Equal objects must have equal hashes** (`a == b` ⇒ `hash(a) == hash(b)`).
- **A key's hash must never change** while it is in the dict — so mutable objects like lists can't be keys.

In [21]:
print("hash('cat') =", hash("cat"), "← changes every time Python starts (string hash randomization)")
print("hash((1, 2)) is stable within a run:", hash((1, 2)) == hash((1, 2)))
print("1 == 1.0 == True and equal hashes:", hash(1) == hash(1.0) == hash(True),
      "→ they are ONE key:", {1: "int", 1.0: "float", True: "bool"})

for candidate in ([1, 2], {"a": 1}, {1, 2}, ([1], 2)):
    try:
        hash(candidate)
    except TypeError as err:
        print(f"{candidate!r:>10} → TypeError: {err}")

hash('cat') = 7787255321723523478 ← changes every time Python starts (string hash randomization)
hash((1, 2)) is stable within a run: True
1 == 1.0 == True and equal hashes: True → they are ONE key: {1: 'bool'}
    [1, 2] → TypeError: unhashable type: 'list'
  {'a': 1} → TypeError: unhashable type: 'dict'
    {1, 2} → TypeError: unhashable type: 'set'
  ([1], 2) → TypeError: unhashable type: 'list'


In [22]:
class PlainToken:                           # default __eq__/__hash__ use identity
    def __init__(self, text):
        self.text = text

print("PlainToken('hi') == PlainToken('hi'):", PlainToken("hi") == PlainToken("hi"))


@dataclass(frozen=True)                     # generates __eq__ AND __hash__ from the fields
class Token:
    text: str
    position: int

vocab = {Token("hi", 0): 101}
print("frozen dataclass as a key:", vocab[Token("hi", 0)])


@dataclass                                  # eq=True without frozen → __hash__ is set to None
class MutableToken:
    text: str

try:
    {MutableToken("hi"): 1}
except TypeError as err:
    print("non-frozen dataclass:", err)

PlainToken('hi') == PlainToken('hi'): False
frozen dataclass as a key: 101
non-frozen dataclass: unhashable type: 'MutableToken'


In [23]:
class BadKey:
    """Hash depends on a MUTABLE attribute — a bug waiting to happen."""
    def __init__(self, value):
        self.value = value
    def __eq__(self, other):
        return isinstance(other, BadKey) and self.value == other.value
    def __hash__(self):
        return hash(self.value)


key = BadKey(1)
lookup = {key: "found me"}
key.value = 2                                             # mutate AFTER inserting
print("same object still in the dict?", key in lookup)    # looks in the bucket for hash(2) → not there
print("dict still holds", len(lookup), "item — now unreachable by key")

same object still in the dict? False
dict still holds 1 item — now unreachable by key


### ✍️ Your Turn

`clicks` records `[user, item]` pairs. Count how many **distinct** pairs there are in O(n). (Lists can't go in a set — what can?)

In [24]:
clicks = [["ana", "item1"], ["ben", "item2"], ["ana", "item1"], ["ana", "item3"], ["ben", "item2"]]
unique_pairs = None  # TODO: an int
check("unique_pairs", unique_pairs, 3, hint="Convert each pair to a tuple, put them in a set, take len().")

⏳ unique_pairs: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
clicks = [["ana", "item1"], ["ben", "item2"], ["ana", "item1"], ["ana", "item3"], ["ben", "item2"]]
unique_pairs = len({tuple(pair) for pair in clicks})
check("unique_pairs", unique_pairs, 3)
```
</details>

> 💡 **Interview angle:** "Why can't a list be a dict key?" — keys are found by hash; a list's contents (and so its hash) could change after insertion, making the key unfindable. Tuples of immutables and frozen dataclasses are hashable. Dict lookup is O(1) on average, O(n) in the worst case (many collisions).

## 8. Time Complexity of Built-ins, Measured 🟡

**Big-O** describes how the cost grows as the input size *n* grows. O(1) = constant, O(n) = grows in step with *n*.

| Operation | list | dict / set | deque |
|---|---|---|---|
| index `x[i]` | O(1) | — | O(n) in the middle |
| `item in x` | **O(n)** | **O(1)** average | O(n) |
| append at end | O(1) amortized | O(1) (`d[k] = v`, `s.add`) | O(1) |
| insert / pop at the **front** | **O(n)** | — | **O(1)** |
| `sorted(x)` | O(n log n) | — | — |

Let's verify two rows instead of trusting the table.

In [25]:
membership = {}
print(f"{'n':>10} | {'x in list':>12} | {'x in set':>10}")
for n in (1_000, 10_000, 100_000, 1_000_000):
    as_list, as_set, target = list(range(n)), set(range(n)), n - 1       # last element: worst case for a list
    t_list = timeit.timeit(lambda: target in as_list, number=20) / 20
    t_set = timeit.timeit(lambda: target in as_set, number=20) / 20
    membership[n] = (t_list, t_set)
    print(f"{n:>10,} | {t_list * 1e6:9.1f} µs | {t_set * 1e6:7.3f} µs")

growth_list = membership[1_000_000][0] / membership[1_000][0]
growth_set = membership[1_000_000][1] / membership[1_000][1]
print(f"\nn grew 1000× → list lookup grew {growth_list:,.0f}×, set lookup grew {growth_set:.1f}×")

         n |    x in list |   x in set
     1,000 |       3.1 µs |   0.048 µs
    10,000 |      28.8 µs |   0.046 µs
   100,000 |     289.6 µs |   0.148 µs
 1,000,000 |    2887.1 µs |   0.104 µs

n grew 1000× → list lookup grew 934×, set lookup grew 2.2×


In [26]:
print(f"{'n':>10} | {'list insert(0)+pop(0)':>22} | {'deque appendleft+popleft':>25}")
for n in (10_000, 100_000, 1_000_000):
    as_list, as_deque = list(range(n)), deque(range(n))
    t_list = timeit.timeit(lambda: (as_list.insert(0, -1), as_list.pop(0)), number=200) / 200
    t_deque = timeit.timeit(lambda: (as_deque.appendleft(-1), as_deque.popleft()), number=200) / 200
    print(f"{n:>10,} | {t_list * 1e6:19.1f} µs | {t_deque * 1e6:22.3f} µs")

         n |  list insert(0)+pop(0) |  deque appendleft+popleft
    10,000 |                 3.2 µs |                  0.037 µs
   100,000 |                35.3 µs |                  0.035 µs
 1,000,000 |               366.0 µs |                  0.035 µs


### ✍️ Your Turn

`common_slow` is O(n·m) because `x in b` scans a list. Write `common_fast` that returns **the same list** (same order as `a`) in O(n + m). It must finish in under 0.5 s on 100,000 + 66,667 items.

In [27]:
catalog = list(range(0, 200_000, 2))        # ids of items in the catalog
purchases = list(range(0, 200_000, 3))      # ids that were bought

def common_slow(a, b):
    return [x for x in a if x in b]         # don't run this on the big lists — it takes minutes

def common_fast(a, b):
    return None  # TODO

with Timer() as t:
    common = common_fast(catalog, purchases)
check("common_fast", None if common is None else (common[:5], len(common), t.seconds < 0.5),
      ([0, 6, 12, 18, 24], 33_334, True), hint="Build set(b) ONCE outside the comprehension, then test `x in that_set`.")

⏳ common_fast: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
catalog = list(range(0, 200_000, 2))
purchases = list(range(0, 200_000, 3))

def common_slow(a, b):
    return [x for x in a if x in b]

def common_fast(a, b):
    b_set = set(b)                      # O(m) once
    return [x for x in a if x in b_set] # O(n) × O(1)

assert common_fast(catalog[:3000], purchases[:2000]) == common_slow(catalog[:3000], purchases[:2000])  # same answer on a small slice
with Timer() as t:
    common = common_fast(catalog, purchases)
check("common_fast", (common[:5], len(common), t.seconds < 0.5), ([0, 6, 12, 18, 24], 33_334, True))
```

Writing `x in set(b)` *inside* the comprehension would rebuild the set for every `x` — back to O(n·m).
</details>

> 💡 **Interview angle:** "Your de-duplication script is slow on 1M rows" — look for `in` on a list or `list.pop(0)` inside a loop. Switch to a `set`/`dict` or `collections.deque`, and *measure* before and after.

## 9. Generators vs Lists: Memory, Measured 🟢

A **list** builds every element up front and keeps them all in memory. A **generator** produces one element at a time, only when asked (**lazy evaluation**), and then forgets it.

- List comprehension: `[x * x for x in range(n)]` → all *n* results exist at once.
- Generator expression: `(x * x for x in range(n))` → one result at a time.
- Generator function: any function with `yield`.

Use generators for streams that are big or endless: reading a huge file line by line, batching a dataset, streaming LLM tokens.

In [28]:
tracemalloc.start()
total_from_list = sum([x * x for x in range(1_000_000)])     # builds all 1M squares first
_, peak_list = tracemalloc.get_traced_memory()
tracemalloc.reset_peak()
total_from_gen = sum(x * x for x in range(1_000_000))        # one square at a time
_, peak_gen = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("same total:", total_from_list == total_from_gen)
print(f"peak memory — list: {peak_list / 1e6:.1f} MB | generator: {peak_gen / 1e3:.1f} KB "
      f"→ {peak_list / peak_gen:,.0f}× less with the generator")

same total: True
peak memory — list: 40.5 MB | generator: 30.0 KB → 1,347× less with the generator


In [29]:
def read_in_batches(items, batch_size):
    """Yield lists of `batch_size` items — the pattern behind every data loader."""
    batch = []
    for item in items:
        print(f"  (reading item {item})")
        batch.append(item)
        if len(batch) == batch_size:
            yield batch          # pause here and hand the batch to the caller
            batch = []
    if batch:
        yield batch


stream = read_in_batches(range(5), batch_size=2)
print("created the generator — nothing was read yet:", type(stream).__name__)
print("first batch :", next(stream))
print("the rest    :", list(stream))
print("second pass :", list(stream), "← a generator can be consumed only once")

created the generator — nothing was read yet: generator
  (reading item 0)
  (reading item 1)
first batch : [0, 1]
  (reading item 2)
  (reading item 3)
  (reading item 4)
the rest    : [[2, 3], [4]]
second pass : [] ← a generator can be consumed only once


### ✍️ Your Turn

Write a **lazy** generator `batches(iterable, size)` that yields lists of `size` items (the last batch may be shorter). The checker also feeds it an *infinite* stream (`itertools.count()`), which only works if you use `yield`.

In [30]:
def batches(iterable, size):
    # TODO: replace this line with a generator that uses `yield`
    return None


got = batches(range(7), 3)
check("batches", None if got is None else (list(got), next(iter(batches(itertools.count(), 2)))),
      ([[0, 1, 2], [3, 4, 5], [6]], [0, 1]), hint="Collect items into a list; yield it when it's full; yield the leftover at the end.")

⏳ batches: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def batches(iterable, size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch


got = batches(range(7), 3)
check("batches", (list(got), next(iter(batches(itertools.count(), 2)))), ([[0, 1, 2], [3, 4, 5], [6]], [0, 1]))
```

Python 3.12 added `itertools.batched(iterable, n)`, which does the same but yields tuples.
</details>

> 💡 **Interview angle:** "List or generator?" — a generator when you only iterate once and the data is large or unbounded (O(1) memory); a list when you need `len()`, indexing, or multiple passes.

## 10. Peeking at Bytecode with dis 🟡

CPython doesn't run your source text directly. It first **compiles** each function to **bytecode** — small instructions for the Python *virtual machine* — and then an interpreter loop executes them one by one. (The cached `.pyc` files in `__pycache__` are exactly this bytecode.) The `dis` module shows them.

Why look? It explains two interview favourites: *why `counter += 1` is not atomic*, and *why the GIL can switch threads in the middle of a line*.

In [31]:
shared_counter = 0

def bump():
    global shared_counter
    shared_counter += 1

dis.dis(bump)
steps = [ins.opname for ins in dis.get_instructions(bump) if ins.opname not in ("RESUME", "RETURN_CONST")]
print("\n`shared_counter += 1` is", len(steps), "separate steps:", steps)

  3           0 RESUME                   0

  5           2 LOAD_GLOBAL              0 (shared_counter)
             12 LOAD_CONST               1 (1)
             14 BINARY_OP               13 (+=)
             18 STORE_GLOBAL             0 (shared_counter)
             20 RETURN_CONST             0 (None)

`shared_counter += 1` is 4 separate steps: ['LOAD_GLOBAL', 'LOAD_CONST', 'BINARY_OP', 'STORE_GLOBAL']


In [32]:
def seconds_per_day():
    return 24 * 60 * 60

dis.dis(seconds_per_day)          # the compiler already multiplied it out: "constant folding"

  1           0 RESUME                   0

  2           2 RETURN_CONST             1 (86400)


In [33]:
SCALE = 3

def use_global(n):
    total = 0
    for i in range(n):
        total += i * SCALE            # LOAD_GLOBAL: dictionary lookup
    return total

def use_local(n, scale=SCALE):
    total = 0
    for i in range(n):
        total += i * scale            # LOAD_FAST: array slot lookup
    return total

print("opcodes used:", sorted({ins.opname for ins in dis.get_instructions(use_global)} & {"LOAD_GLOBAL", "LOAD_FAST"}),
      "vs", sorted({ins.opname for ins in dis.get_instructions(use_local)} & {"LOAD_GLOBAL", "LOAD_FAST"}))
t_global = min(timeit.repeat(lambda: use_global(200_000), number=5, repeat=3))
t_local = min(timeit.repeat(lambda: use_local(200_000), number=5, repeat=3))
ratio = t_global / t_local
print(f"global: {t_global:.3f}s | local: {t_local:.3f}s | global/local = {ratio:.2f}")
if ratio > 1.15:
    print("→ local lookups were noticeably faster here.")
else:
    print("→ the difference is small: since Python 3.11 the interpreter specializes and caches global lookups, "
          "so this old micro-optimization rarely matters.")

opcodes used: ['LOAD_FAST', 'LOAD_GLOBAL'] vs ['LOAD_FAST', 'LOAD_GLOBAL']
global: 0.023s | local: 0.022s | global/local = 1.06
→ the difference is small: since Python 3.11 the interpreter specializes and caches global lookups, so this old micro-optimization rarely matters.


> 💡 **Interview angle:** "Is Python compiled or interpreted?" — both: source is compiled to bytecode, which the CPython virtual machine interprets (3.11+ specializes hot instructions; 3.13 added an experimental JIT). `counter += 1` is several bytecodes, so another thread can run between the read and the write.

## 11. Threads and the GIL 🟡

A **thread** is a separate line of execution inside the same process. Threads share memory — easy to pass data, easy to corrupt it.

The **GIL (Global Interpreter Lock)** is a lock inside CPython: **only one thread can execute Python bytecode at a time**.

- **What it protects:** CPython's internals — above all the reference counts from Section 2. Without it, two threads updating the same count could free an object that is still in use.
- **What it does *not* protect:** *your* data. `counter += 1` is several bytecodes, and a thread switch can happen between them.
- **When threads still help:** the GIL is **released while waiting** (sleep, network, disk) and inside many C extensions (NumPy, `hashlib`, PyTorch ops). So threads speed up **I/O-bound** work, but not **CPU-bound pure-Python** work.

**I/O-bound** = the task mostly *waits* (HTTP calls, database queries, reading files). **CPU-bound** = the task mostly *computes* (loops, parsing, math in Python).

### Experiment 1 — I/O-bound work with threads

In [34]:
def fake_io(job_id):
    time.sleep(0.2)              # stands in for a network call; sleeping releases the GIL
    return job_id


with Timer() as t_seq:
    seq_results = [fake_io(i) for i in range(8)]

thread_results = [None] * 8

def run_job(i):
    thread_results[i] = fake_io(i)

with Timer() as t_thr:
    workers_list = [threading.Thread(target=run_job, args=(i,)) for i in range(8)]
    for th in workers_list:
        th.start()
    for th in workers_list:
        th.join()                # wait for every thread to finish

assert thread_results == seq_results
print(f"8 × 0.2 s I/O jobs → sequential {t_seq.seconds:.2f} s | 8 threads {t_thr.seconds:.2f} s "
      f"→ {t_seq.seconds / t_thr.seconds:.1f}× faster")

8 × 0.2 s I/O jobs → sequential 1.66 s | 8 threads 0.21 s → 7.9× faster


### Experiment 2 — CPU-bound pure-Python work with threads

In [35]:
def count_primes(limit):
    """Pure-Python CPU work: count primes below `limit` by trial division."""
    count = 0
    for n in range(2, limit):
        d = 2
        while d * d <= n:
            if n % d == 0:
                break
            d += 1
        else:
            count += 1
    return count


PRIME_LIMIT = 150_000
N_JOBS = 4

with Timer() as t_cpu_seq:
    cpu_seq = [count_primes(PRIME_LIMIT) for _ in range(N_JOBS)]

cpu_thr = [None] * N_JOBS

def run_cpu_job(i):
    cpu_thr[i] = count_primes(PRIME_LIMIT)

with Timer() as t_cpu_thr:
    cpu_threads = [threading.Thread(target=run_cpu_job, args=(i,)) for i in range(N_JOBS)]
    for th in cpu_threads:
        th.start()
    for th in cpu_threads:
        th.join()

assert cpu_thr == cpu_seq
cpu_thread_speedup = t_cpu_seq.seconds / t_cpu_thr.seconds
print(f"{N_JOBS} × count_primes({PRIME_LIMIT:,}) = {cpu_seq[0]:,} primes each")
print(f"sequential {t_cpu_seq.seconds:.2f} s | {N_JOBS} threads {t_cpu_thr.seconds:.2f} s → speedup {cpu_thread_speedup:.2f}×")
if cpu_thread_speedup < 1.5:
    print("→ no real speedup: the threads took turns holding the GIL, so the CPU work ran one thread at a time.")
else:
    print("→ threads ran in parallel — this interpreter is not limited by a GIL.")

4 × count_primes(150,000) = 13,848 primes each
sequential 0.42 s | 4 threads 0.39 s → speedup 1.07×
→ no real speedup: the threads took turns holding the GIL, so the CPU work ran one thread at a time.


### Experiment 3 — CPU-bound work in C code that releases the GIL

`hashlib` releases the GIL while hashing large buffers, so here threads *do* use several cores even though the work is CPU-bound. NumPy and PyTorch behave the same way for most heavy operations.

In [36]:
blob = os.urandom(64 * 1024 * 1024)                 # 64 MB of random bytes

def sha256_hex(buffer):
    return hashlib.sha256(buffer).hexdigest()

with Timer() as t_hash_seq:
    digests_seq = [sha256_hex(blob) for _ in range(N_JOBS)]
with Timer() as t_hash_thr:
    with ThreadPoolExecutor(max_workers=N_JOBS) as pool:        # a thread pool — explained in Section 12
        digests_thr = list(pool.map(sha256_hex, [blob] * N_JOBS))

assert digests_seq == digests_thr
hash_speedup = t_hash_seq.seconds / t_hash_thr.seconds
print(f"hash 64 MB × {N_JOBS} → sequential {t_hash_seq.seconds:.2f} s | {N_JOBS} threads {t_hash_thr.seconds:.2f} s "
      f"→ speedup {hash_speedup:.1f}×")
if hash_speedup > 1.5:
    print("→ threads used several cores: hashlib released the GIL while hashing.")
else:
    print("→ little speedup on this machine (hashing may be too fast for thread overhead to pay off).")
del blob

hash 64 MB × 4 → sequential 0.08 s | 4 threads 0.02 s → speedup 3.8×
→ threads used several cores: hashlib released the GIL while hashing.


### Experiment 4 — The GIL does not make your code thread-safe

A **race condition** happens when the result depends on the unlucky timing of threads. Below, each deposit does *read → (thread switch) → write*. `time.sleep(0)` only makes the switch likely; in real code any I/O, logging call, or simply bad luck does the same. A **`threading.Lock`** lets only one thread run the critical section at a time.

In [37]:
balance = 0

def deposit_unsafe(times):
    global balance
    for _ in range(times):
        current = balance          # 1. read
        time.sleep(0)              # 2. another thread may run here
        balance = current + 1      # 3. write — can overwrite someone else's deposit

def run_in_threads(target, n_threads=8, times=2_000):
    threads = [threading.Thread(target=target, args=(times,)) for _ in range(n_threads)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    return n_threads * times

expected_total = run_in_threads(deposit_unsafe)
print(f"❌ no lock  : balance {balance:,} of {expected_total:,} → {expected_total - balance:,} deposits lost")

balance = 0
balance_lock = threading.Lock()

def deposit_safe(times):
    global balance
    for _ in range(times):
        with balance_lock:         # only one thread at a time inside this block
            current = balance
            time.sleep(0)
            balance = current + 1

expected_total = run_in_threads(deposit_safe)
print(f"✅ with lock: balance {balance:,} of {expected_total:,}")

❌ no lock  : balance 2,041 of 16,000 → 13,959 deposits lost
✅ with lock: balance 16,000 of 16,000


### 🔬 Advanced note: free-threaded CPython (3.13+)

[PEP 703](https://peps.python.org/pep-0703/) makes the GIL optional. Python **3.13** shipped a separate, experimental *free-threaded* build (often installed as `python3.13t`), and in **3.14** that build became officially supported — but it is still a **separate, opt-in build**; the default `python` keeps the GIL. On a free-threaded build, Experiment 2 *can* speed up with threads, `sys._is_gil_enabled()` returns `False`, and unprotected code like `deposit_unsafe` loses updates even more readily. Libraries with C extensions must be rebuilt to support it, so in 2025–2026 most production AI stacks still assume a GIL. Knowing this nuance is a strong interview signal.

> 💡 **Interview angle:** "What is the GIL and when does it hurt?" — a CPython lock allowing one thread to run bytecode at a time, protecting interpreter internals like refcounts. It hurts CPU-bound pure-Python threads; it doesn't hurt I/O-bound threads or C extensions that release it. It does **not** make your own read-modify-write code thread-safe — use a `Lock` or a `queue.Queue`.

## 12. Processes and concurrent.futures 🟡

A **process** is a separate running program with **its own memory and its own interpreter — and so its own GIL**. Several processes can run Python code on several cores at once. The costs:

1. **Start-up:** each worker process must start a Python interpreter.
2. **Pickling:** arguments and results are serialized (`pickle`) and copied between processes.
3. **No shared variables:** a worker can't see or change your notebook's variables.

### How a new process starts: `spawn` vs `fork`

| Start method | What happens | Default on |
|---|---|---|
| `fork` | copies the parent process as it is | older Linux Pythons (until 3.13) |
| `forkserver` | forks from a clean helper process | Linux from Python 3.14 |
| `spawn` | starts a **fresh** interpreter that **imports** your code | **macOS and Windows** |

**The Jupyter gotcha:** with `spawn`, the worker receives your function *by name* (module + function name) and imports it. A function defined in a notebook cell lives in the kernel's special `__main__` module, which a fresh process cannot import → `AttributeError: Can't get attribute ...` and a `BrokenProcessPool` error. Lambdas can't be pickled at all.

**The fix:** put worker functions in a real `.py` file and import them. We write a small module into `_outputs/` and add that folder to `sys.path` (child processes inherit `sys.path`, so they find it too).

In [38]:
WORKER_SOURCE = '''"""Worker functions for ProcessPoolExecutor.
They live in a real module so that spawned child processes can import them by name."""


def count_primes(limit):
    """Pure-Python CPU work: count primes below `limit` by trial division."""
    return count_primes_in_range((2, limit))


def count_primes_in_range(bounds):
    """Count primes p with start <= p < stop; `bounds` is a (start, stop) tuple."""
    start, stop = bounds
    count = 0
    for n in range(max(start, 2), stop):
        d = 2
        while d * d <= n:
            if n % d == 0:
                break
            d += 1
        else:
            count += 1
    return count


def square(x):
    return x * x
'''

worker_file = OUTPUT_DIR / "concurrency_workers.py"
worker_file.write_text(WORKER_SOURCE)
worker_dir = str(OUTPUT_DIR.resolve())
if worker_dir not in sys.path:
    sys.path.insert(0, worker_dir)
importlib.invalidate_caches()

import concurrency_workers as workers
workers = importlib.reload(workers)                 # pick up edits if you re-run this cell
assert workers.count_primes(10_000) == count_primes(10_000)
print("worker module:", Path(workers.__file__).relative_to(Path.cwd()))

worker module: _outputs/concurrency_workers.py


### The same CPU-bound benchmark: sequential vs threads vs processes

`concurrent.futures` gives threads and processes **one identical interface**: an *executor* with `map` and `submit`. Swapping `ThreadPoolExecutor` for `ProcessPoolExecutor` is a one-word change. The process timing below *includes* starting the worker processes.

In [39]:
jobs = [PRIME_LIMIT] * N_JOBS

with Timer() as t_seq:
    counts_seq = [workers.count_primes(n) for n in jobs]
with Timer() as t_threads:
    with ThreadPoolExecutor(max_workers=N_JOBS) as pool:
        counts_threads = list(pool.map(workers.count_primes, jobs))
with Timer() as t_procs:
    with ProcessPoolExecutor(max_workers=N_JOBS) as pool:
        counts_procs = list(pool.map(workers.count_primes, jobs))

assert counts_seq == counts_threads == counts_procs
cpu_table = {"sequential": t_seq.seconds, "threads": t_threads.seconds, "processes": t_procs.seconds}
for label, secs in cpu_table.items():
    print(f"{label:<11} {secs:5.2f} s   speedup {t_seq.seconds / secs:4.2f}×")
best = min(cpu_table, key=cpu_table.get)
print(f"→ fastest for CPU-bound pure Python: {best}")

sequential   0.40 s   speedup 1.00×
threads      0.40 s   speedup 1.02×
processes    0.16 s   speedup 2.56×
→ fastest for CPU-bound pure Python: processes


### Processes have overhead: tiny tasks get *slower*

Every task is pickled, sent to a worker, run, and its result pickled back. For microscopic tasks that round trip costs far more than the work. `chunksize` sends many items per trip.

In [40]:
tiny_inputs = list(range(5_000))
with Timer() as t_tiny_seq:
    tiny_seq = [workers.square(v) for v in tiny_inputs]
with ProcessPoolExecutor(max_workers=N_JOBS) as pool:
    list(pool.map(workers.square, range(N_JOBS)))       # warm up: start the workers first
    with Timer() as t_tiny_chunk1:
        tiny_chunk1 = list(pool.map(workers.square, tiny_inputs))
    with Timer() as t_tiny_chunk500:
        tiny_chunk500 = list(pool.map(workers.square, tiny_inputs, chunksize=500))

assert tiny_seq == tiny_chunk1 == tiny_chunk500
print(f"5,000 tiny squares → sequential {t_tiny_seq.seconds * 1000:.1f} ms | processes chunksize=1 "
      f"{t_tiny_chunk1.seconds * 1000:.0f} ms | chunksize=500 {t_tiny_chunk500.seconds * 1000:.0f} ms")
print(f"→ chunksize=1 was {t_tiny_chunk1.seconds / t_tiny_seq.seconds:,.0f}× slower than a plain loop; "
      f"batching cut the process time by {t_tiny_chunk1.seconds / t_tiny_chunk500.seconds:.0f}×")

5,000 tiny squares → sequential 0.1 ms | processes chunksize=1 297 ms | chunksize=500 1 ms
→ chunksize=1 was 1,997× slower than a plain loop; batching cut the process time by 251×


### `submit` + `as_completed`: handle results as they finish, and catch worker errors

`pool.map` returns results in input order. `pool.submit` returns a **Future** — a placeholder for a result that isn't ready yet — and `as_completed` yields futures in the order they **finish**. An exception raised in a worker is re-raised when you call `future.result()`.

In [41]:
def flaky_job(job_id):
    time.sleep(0.05 * (5 - job_id))          # later jobs finish sooner
    if job_id == 3:
        raise ValueError(f"job {job_id} failed")
    return job_id * 10

with ThreadPoolExecutor(max_workers=5) as pool:
    future_to_job = {pool.submit(flaky_job, i): i for i in range(5)}
    for future in as_completed(future_to_job):
        job_id = future_to_job[future]
        try:
            print(f"job {job_id} finished → {future.result()}")
        except ValueError as err:
            print(f"job {job_id} raised   → {err!r}")

job 4 finished → 40
job 3 raised   → ValueError('job 3 failed')
job 2 finished → 20


job 1 finished → 10
job 0 finished → 0


### ✍️ Your Turn

`slow_lookup` takes 0.1 s (think: a database or HTTP call). Use a `ThreadPoolExecutor` to look up all 10 users **in the original order**, in under 0.5 s total.

In [42]:
def slow_lookup(user_id):
    time.sleep(0.1)
    return f"user-{user_id}"

user_ids = list(range(10))
with Timer() as t:
    names = None  # TODO
check("thread_pool_lookup", None if names is None else (names, t.seconds < 0.5),
      ([f"user-{i}" for i in range(10)], True), hint="with ThreadPoolExecutor(max_workers=10) as pool: names = list(pool.map(...))")

⏳ thread_pool_lookup: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def slow_lookup(user_id):
    time.sleep(0.1)
    return f"user-{user_id}"

user_ids = list(range(10))
with Timer() as t:
    with ThreadPoolExecutor(max_workers=10) as pool:
        names = list(pool.map(slow_lookup, user_ids))
check("thread_pool_lookup", (names, t.seconds < 0.5), ([f"user-{i}" for i in range(10)], True))
```

Threads are the right tool: the work is waiting, and the function is defined in the notebook (threads share the kernel's memory, so no importable module is needed).
</details>

> 💡 **Interview angle:** "Threads or processes?" — processes for CPU-bound pure Python (separate GILs), threads for I/O-bound work (cheap, shared memory). Mention the costs of processes (start-up, pickling, no shared state) and that with `spawn` the worker function must be importable.

## 13. asyncio: async, await, gather, Semaphores, Timeouts 🔴

`asyncio` runs many I/O tasks on **one thread** using an **event loop** — a scheduler that keeps a list of paused tasks and resumes whichever one is ready.

- `async def f():` defines a **coroutine function**. Calling `f()` does **not** run it; it creates a *coroutine object*.
- `await something` means *"pause me here until this is done, and let the loop run other tasks meanwhile."*
- Tasks switch **only at `await`** (cooperative multitasking). A task that never awaits — `time.sleep`, a long CPU loop, a non-async HTTP library — **blocks every other task**.

| | Threads | asyncio |
|---|---|---|
| Who decides when to switch? | the OS, at any time | *you*, at each `await` |
| Cost per task | ~MBs of stack, OS thread | ~KBs, a Python object |
| Scales to | hundreds | tens of thousands of connections |
| Needs async libraries? | no | **yes** (`httpx.AsyncClient`, `aiohttp`, async SDK clients) |

**The Jupyter gotcha:** a Jupyter kernel is *already running* an event loop, so you use **top-level `await`** directly in a cell. In a normal `.py` script you write `asyncio.run(main())` instead — and calling `asyncio.run` inside Jupyter fails:

In [43]:
async def download(name, seconds):
    await asyncio.sleep(seconds)          # non-blocking wait: the loop runs other tasks meanwhile
    return f"{name} done"

coro = download("test", 0.01)
print("calling an async function returns:", type(coro).__name__, "(nothing has run yet)")
try:
    asyncio.run(coro)                     # what you'd write in a script
except RuntimeError as err:
    coro.close()                          # tidy up the coroutine that never ran
    print("asyncio.run inside Jupyter → RuntimeError:", err)

print("top-level await works in Jupyter →", await download("test", 0.01))

calling an async function returns: coroutine (nothing has run yet)
asyncio.run inside Jupyter → RuntimeError: asyncio.run() cannot be called from a running event loop
top-level await works in Jupyter → test done


In [44]:
with Timer() as t_one_by_one:
    one_by_one = [await download("a", 0.3), await download("b", 0.3), await download("c", 0.3)]

with Timer() as t_gather:
    together = await asyncio.gather(download("a", 0.3), download("b", 0.3), download("c", 0.3))

print(f"awaiting one by one: {t_one_by_one.seconds:.2f} s → {one_by_one}")
print(f"asyncio.gather     : {t_gather.seconds:.2f} s → {together}  (results keep the input order)")

awaiting one by one: 0.91 s → ['a done', 'b done', 'c done']
asyncio.gather     : 0.30 s → ['a done', 'b done', 'c done']  (results keep the input order)


### Limiting concurrency with a Semaphore

Firing 10,000 requests at once will get you rate-limited, run out of sockets, or crash a server. An `asyncio.Semaphore(n)` is a counter of *n* permits: `async with sem:` takes a permit (waiting if none are free) and gives it back at the end.

In [45]:
async def call_service(i, limiter, stats):
    async with limiter:                               # at most N coroutines inside this block
        stats["in_flight"] += 1
        stats["peak"] = max(stats["peak"], stats["in_flight"])
        await asyncio.sleep(0.1)
        stats["in_flight"] -= 1
        return i

stats = {"in_flight": 0, "peak": 0}
limiter = asyncio.Semaphore(3)
with Timer() as t_sem:
    served = await asyncio.gather(*(call_service(i, limiter, stats) for i in range(12)))

print(f"12 calls of 0.1 s with Semaphore(3): peak in flight = {stats['peak']}, took {t_sem.seconds:.2f} s, "
      f"order kept = {served == list(range(12))}")

12 calls of 0.1 s with Semaphore(3): peak in flight = 3, took 0.41 s, order kept = True


### Timeouts, errors, and TaskGroup

- `async with asyncio.timeout(s):` (Python 3.11+) or `asyncio.wait_for(coro, s)` cancels work that takes too long and raises `TimeoutError`.
- `gather(..., return_exceptions=True)` returns exceptions as values instead of failing on the first one.
- `asyncio.TaskGroup` (3.11+) is *structured concurrency*: if one task fails, the others are cancelled and the error is raised.

In [46]:
async def slow_model(seconds):
    await asyncio.sleep(seconds)
    return f"answered in {seconds} s"

try:
    async with asyncio.timeout(0.2):
        await slow_model(1.0)
except TimeoutError:
    print("asyncio.timeout: gave up after 0.2 s")

outcomes = await asyncio.gather(*(asyncio.wait_for(slow_model(s), timeout=0.25) for s in (0.1, 0.5, 0.2)),
                                return_exceptions=True)
for s, outcome in zip((0.1, 0.5, 0.2), outcomes):
    print(f"  task needing {s} s → {type(outcome).__name__}: {outcome}")

async with asyncio.TaskGroup() as tg:
    first = tg.create_task(slow_model(0.1))
    second = tg.create_task(slow_model(0.2))
print("TaskGroup results:", first.result(), "|", second.result())

asyncio.timeout: gave up after 0.2 s


  task needing 0.1 s → str: answered in 0.1 s
  task needing 0.5 s → TimeoutError: 
  task needing 0.2 s → str: answered in 0.2 s


TaskGroup results: answered in 0.1 s | answered in 0.2 s


### Escape hatch: running blocking code from async code

If you must call a blocking function (an old SDK, a file read), `await asyncio.to_thread(fn, *args)` runs it in a worker thread so the event loop keeps going.

In [47]:
def blocking_read(i):
    time.sleep(0.2)          # a blocking call we can't rewrite
    return i

with Timer() as t_to_thread:
    via_threads = await asyncio.gather(*(asyncio.to_thread(blocking_read, i) for i in range(5)))
print(f"5 × blocking 0.2 s calls via asyncio.to_thread: {t_to_thread.seconds:.2f} s → {via_threads}")

5 × blocking 0.2 s calls via asyncio.to_thread: 0.21 s → [0, 1, 2, 3, 4]


### ✍️ Your Turn

Run `fetch_page` for pages 1–5 **concurrently** and collect the results in page order. It must take less than 0.4 s in total.

In [48]:
async def fetch_page(page):
    await asyncio.sleep(0.2)
    return page * 10

with Timer() as t:
    pages = None  # TODO: use await asyncio.gather(...)
check("gather_pages", None if pages is None else (list(pages), t.seconds < 0.4), ([10, 20, 30, 40, 50], True),
      hint="pages = await asyncio.gather(*(fetch_page(p) for p in range(1, 6)))")

⏳ gather_pages: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
async def fetch_page(page):
    await asyncio.sleep(0.2)
    return page * 10

with Timer() as t:
    pages = await asyncio.gather(*(fetch_page(p) for p in range(1, 6)))
check("gather_pages", (list(pages), t.seconds < 0.4), ([10, 20, 30, 40, 50], True))
```

In a script you would wrap this in `async def main(): ...` and call `asyncio.run(main())`.
</details>

> 💡 **Interview angle:** "asyncio vs threads?" — both suit I/O-bound work; asyncio is one thread with explicit switch points, scales to many thousands of connections cheaply and avoids most race conditions, but needs async-aware libraries, and any blocking call freezes everything. Neither helps CPU-bound Python.

## 14. Concurrency in AI Work 🔴

### Choosing the tool

| AI workload | Bound by | Use | Why |
|---|---|---|---|
| Calling LLM / embedding APIs for many prompts | waiting (network) | `asyncio` + `Semaphore` + retries (or a thread pool) | thousands of cheap concurrent waits, controlled rate |
| Downloading datasets, images, web pages | waiting | threads or `asyncio` | same as above |
| Tokenizing / cleaning / augmenting data in **pure Python** | CPU | processes: `ProcessPoolExecutor`, `datasets.map(num_proc=...)`, `DataLoader(num_workers=...)` | each process has its own GIL |
| NumPy / PyTorch math on arrays | CPU in C/GPU | vectorize; the library already multithreads | these release the GIL internally; extra Python threads add little |
| Serving a model behind an API | both | async web framework for I/O; model calls in a thread/process pool or a separate inference server | never block the event loop |

### PyTorch `DataLoader` workers

`DataLoader(dataset, num_workers=4)` starts **4 worker processes** that call `dataset[i]` in parallel and send batches back — processes precisely because image decoding and augmentation are CPU-bound Python. Everything from Section 12 applies:

- On macOS and Windows workers use `spawn`, so your `Dataset` class must be importable (defined in a `.py` file, not only in a notebook cell), and training scripts need an `if __name__ == "__main__":` guard.
- Worker start-up is not free. Use `persistent_workers=True` so workers are not re-created every epoch, and for small, fast datasets measure `num_workers=0` too — it can win.
- Libraries that are already parallel inside (e.g. Hugging Face `tokenizers`, written in Rust) don't need Python processes on top; mixing both can oversubscribe CPU cores.

```python
# train.py
from torch.utils.data import DataLoader
from my_project.data import ImageDataset          # importable module → works with spawn

if __name__ == "__main__":                        # required with spawn
    loader = DataLoader(ImageDataset("data/"), batch_size=64, num_workers=4, persistent_workers=True)
```

### Calling an LLM API concurrently without getting rate-limited

We can't call a real paid API here, so we use a small **local stand-in** for one (no network, no key): each call takes 0.1 s, and — like a real provider returning **HTTP 429 Too Many Requests** — it rejects calls when more than 5 are in flight. Real providers limit *requests and tokens per minute*, and their official Python SDKs (`openai`, `anthropic`) already retry 429s with exponential backoff (2 retries by default); the patterns below are what you add on top when you send thousands of requests.

In [49]:
class RateLimitError(Exception):
    """What an API client raises for HTTP 429 Too Many Requests."""


class LocalLLMStandIn:
    """Local stand-in for a hosted LLM API: fixed latency, rejects calls beyond `max_concurrent` in flight."""

    def __init__(self, max_concurrent=5, latency=0.1):
        self.max_concurrent, self.latency = max_concurrent, latency
        self.in_flight = self.rejected = 0

    async def complete(self, prompt):
        if self.in_flight >= self.max_concurrent:
            self.rejected += 1
            raise RateLimitError("429 Too Many Requests")
        self.in_flight += 1
        try:
            await asyncio.sleep(self.latency)
            return f"summary of {prompt}"
        finally:
            self.in_flight -= 1


async def complete_with_retry(api, prompt, limiter, max_retries=6, base_delay=0.05):
    """Semaphore for concurrency + exponential backoff with jitter for 429s."""
    for attempt in range(max_retries + 1):
        try:
            async with limiter:
                return await api.complete(prompt)
        except RateLimitError:
            if attempt == max_retries:
                raise
            await asyncio.sleep(base_delay * 2 ** attempt * random.uniform(0.5, 1.5))


async def run_strategy(name, make_call, api, n_prompts=40):
    prompts = [f"document-{i}" for i in range(n_prompts)]
    with Timer() as timer:
        outcomes = await asyncio.gather(*(make_call(api, p) for p in prompts), return_exceptions=True)
    succeeded = sum(not isinstance(o, Exception) for o in outcomes)
    return {"strategy": name, "succeeded": succeeded, "failed": n_prompts - succeeded,
            "429s seen": api.rejected, "seconds": timer.seconds}


limit_5 = asyncio.Semaphore(5)
limit_8 = asyncio.Semaphore(8)                      # deliberately above the server's limit

async def fire_everything(api, p):
    return await api.complete(p)

async def with_semaphore(api, p):
    async with limit_5:
        return await api.complete(p)

async def semaphore_and_retry(api, p):
    return await complete_with_retry(api, p, limit_8)

llm_results = [
    await run_strategy("gather everything at once", fire_everything, LocalLLMStandIn()),
    await run_strategy("Semaphore(5)", with_semaphore, LocalLLMStandIn()),
    await run_strategy("Semaphore(8) + retry/backoff", semaphore_and_retry, LocalLLMStandIn()),
]
print(f"{'strategy':<30} {'ok':>4} {'failed':>7} {'429s':>5} {'seconds':>8}")
for r in llm_results:
    print(f"{r['strategy']:<30} {r['succeeded']:>4} {r['failed']:>7} {r['429s seen']:>5} {r['seconds']:>8.2f}")
sequential_estimate = 40 * 0.1
winner = min((r for r in llm_results if r["failed"] == 0), key=lambda r: r["seconds"])
print(f"\n→ sequential calls would need ≈{sequential_estimate:.1f} s. Fastest strategy with zero failures: "
      f"'{winner['strategy']}' ({winner['seconds']:.2f} s).")

strategy                         ok  failed  429s  seconds
gather everything at once         5      35    35     0.10
Semaphore(5)                     40       0     0     0.81
Semaphore(8) + retry/backoff     40       0   119     1.78

→ sequential calls would need ≈4.0 s. Fastest strategy with zero failures: 'Semaphore(5)' (0.81 s).


**What the numbers show:** firing everything at once finishes fast only because most calls *fail*. A semaphore sized to the provider's limit gives zero errors; when you don't know the limit (or share it with other services), retries with **exponential backoff + jitter** recover from 429s — the random jitter stops all clients retrying at the same instant.

> 💡 **Interview angle:** "Embed 1M documents with a rate-limited API" — async client + semaphore sized to the limit, batch many texts per request, retry 429/5xx with exponential backoff and jitter, respect `Retry-After`, checkpoint progress so a crash doesn't restart from zero, and track tokens per minute, not just requests.

## 🔧 Build It From Scratch

Three classics interviewers ask for, each checked against the standard library.

### 1) A tiny thread pool with `queue.Queue`

A **thread pool** keeps a few long-lived worker threads that pull jobs from a shared queue. `queue.Queue` is already thread-safe (it uses locks internally), so workers can share it without extra locking. We send one *sentinel* object per worker to tell it to stop.

In [50]:
class TinyFuture:
    """A placeholder for a result that another thread will fill in."""

    def __init__(self):
        self._done = threading.Event()
        self._result = self._error = None

    def set_result(self, value):
        self._result = value
        self._done.set()

    def set_error(self, error):
        self._error = error
        self._done.set()

    def result(self, timeout=None):
        if not self._done.wait(timeout):
            raise TimeoutError("result not ready")
        if self._error is not None:
            raise self._error
        return self._result


class TinyThreadPool:
    _STOP = object()                                   # sentinel: "no more work"

    def __init__(self, n_workers):
        self._jobs = queue.Queue()
        self._threads = [threading.Thread(target=self._work, daemon=True) for _ in range(n_workers)]
        for th in self._threads:
            th.start()

    def _work(self):
        while True:
            job = self._jobs.get()                     # blocks until a job arrives
            if job is self._STOP:
                return
            fn, args, future = job
            try:
                future.set_result(fn(*args))
            except Exception as err:                   # hand worker errors to the caller
                future.set_error(err)

    def submit(self, fn, *args):
        future = TinyFuture()
        self._jobs.put((fn, args, future))
        return future

    def map(self, fn, items):
        futures = [self.submit(fn, item) for item in items]
        return [f.result() for f in futures]           # input order, like Executor.map

    def shutdown(self):
        for _ in self._threads:
            self._jobs.put(self._STOP)
        for th in self._threads:
            th.join()

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        self.shutdown()


def slow_double(x):
    time.sleep(0.1)
    return 2 * x

with Timer() as t_tiny_pool:
    with TinyThreadPool(n_workers=4) as tiny_pool:
        doubled = tiny_pool.map(slow_double, range(8))
        failing = tiny_pool.submit(lambda: 1 / 0)
        try:
            failing.result()
        except ZeroDivisionError as err:
            error_seen = repr(err)

with ThreadPoolExecutor(max_workers=4) as reference_pool:
    expected_doubled = list(reference_pool.map(slow_double, range(8)))

assert doubled == expected_doubled == [2 * x for x in range(8)]
assert error_seen.startswith("ZeroDivisionError")
assert all(not th.is_alive() for th in tiny_pool._threads)
print(f"✅ TinyThreadPool matches ThreadPoolExecutor: {doubled}")
print(f"   8 × 0.1 s jobs on 4 workers took {t_tiny_pool.seconds:.2f} s (sequential ≈ 0.8 s); worker error surfaced as {error_seen}")

✅ TinyThreadPool matches ThreadPoolExecutor: [0, 2, 4, 6, 8, 10, 12, 14]
   8 × 0.1 s jobs on 4 workers took 0.21 s (sequential ≈ 0.8 s); worker error surfaced as ZeroDivisionError('division by zero')


### 2) An async rate-limited fetcher

Real APIs limit **concurrency** *and* **requests per time window**. Our limiter enforces both without using `asyncio.Semaphore`:

- a counter of in-flight calls plus an `asyncio.Condition` so waiters wake up when a slot frees, and
- a **sliding window**: remember the start times of recent calls; if `rate` calls started within the last `per` seconds, sleep until the oldest one leaves the window.

In [51]:
class AsyncRateLimiter:
    def __init__(self, max_concurrent, rate, per):
        self.max_concurrent, self.rate, self.per = max_concurrent, rate, per
        self._in_flight = 0
        self._starts = deque()
        self._condition = asyncio.Condition()

    async def __aenter__(self):
        while True:
            async with self._condition:
                now = time.monotonic()
                while self._starts and now - self._starts[0] >= self.per:
                    self._starts.popleft()                         # forget starts outside the window
                if self._in_flight < self.max_concurrent and len(self._starts) < self.rate:
                    self._in_flight += 1
                    self._starts.append(now)
                    return self
                if self._in_flight >= self.max_concurrent:
                    await self._condition.wait()                   # releases the lock until a call finishes
                    continue
                wait = self.per - (now - self._starts[0])          # time until the oldest start leaves the window
            await asyncio.sleep(wait)                              # sleep OUTSIDE the lock

    async def __aexit__(self, *exc_info):
        async with self._condition:
            self._in_flight -= 1
            self._condition.notify_all()


async def fetch_all(urls, fetch_one, limiter):
    async def guarded(url):
        async with limiter:
            return await fetch_one(url)
    return await asyncio.gather(*(guarded(u) for u in urls))


# Verify the guarantees with a local stand-in that records what happens
log = {"in_flight": 0, "peak": 0, "starts": []}

async def recording_fetch(url):
    log["starts"].append(time.monotonic())
    log["in_flight"] += 1
    log["peak"] = max(log["peak"], log["in_flight"])
    await asyncio.sleep(0.05)
    log["in_flight"] -= 1
    return url.upper()

fake_urls = [f"https://example.com/item/{i}" for i in range(12)]
limiter = AsyncRateLimiter(max_concurrent=3, rate=4, per=0.25)
with Timer() as t_limited:
    fetched = await fetch_all(fake_urls, recording_fetch, limiter)

starts = sorted(log["starts"])
busiest_window = max(sum(1 for s in starts if first <= s < first + 0.25 - 1e-3) for first in starts)
assert fetched == [u.upper() for u in fake_urls]
assert log["peak"] <= 3, log["peak"]
assert busiest_window <= 4, busiest_window
print(f"✅ results in order | peak in flight {log['peak']} (limit 3) | most starts in any 0.25 s window: "
      f"{busiest_window} (limit 4) | took {t_limited.seconds:.2f} s")

✅ results in order | peak in flight 3 (limit 3) | most starts in any 0.25 s window: 4 (limit 4) | took 0.61 s


Now the same limiter on **real HTTP requests** — a few small CSV files from the public [seaborn-data repository](https://github.com/mwaskom/seaborn-data) on GitHub.

In [52]:
SEABORN_DATA = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/{}.csv"
real_urls = [SEABORN_DATA.format(name) for name in ("tips", "iris", "penguins", "flights", "exercise", "anscombe")]

try:
    async with httpx.AsyncClient(timeout=15) as client:
        async def fetch_bytes(url):
            response = await client.get(url)
            response.raise_for_status()
            return len(response.content)

        with Timer() as t_real:
            sizes = await fetch_all(real_urls, fetch_bytes, AsyncRateLimiter(max_concurrent=3, rate=4, per=0.5))
    for url, size in zip(real_urls, sizes):
        print(f"  {url.rsplit('/', 1)[-1]:<14} {size:>7,} bytes")
    print(f"✅ fetched {len(sizes)} real files through the limiter in {t_real.seconds:.2f} s")
except httpx.HTTPError as err:
    print(f"⏭️ Skipped: could not reach GitHub ({err!r}). The limiter itself was verified above.")

  tips.csv         9,729 bytes
  iris.csv         3,858 bytes
  penguins.csv    13,478 bytes
  flights.csv      2,350 bytes
  exercise.csv     2,735 bytes
  anscombe.csv       556 bytes
✅ fetched 6 real files through the limiter in 0.53 s


### 3) An LRU cache with `OrderedDict`

An **LRU (Least Recently Used) cache** keeps the `capacity` most recently used items; when full, it evicts the item unused for the longest time. `OrderedDict` remembers order and can move a key to the end in O(1), so both `get` and `put` are O(1).

In [53]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self._data = OrderedDict()
        self.hits = self.misses = 0

    def get(self, key, default=None):
        if key not in self._data:
            self.misses += 1
            return default
        self.hits += 1
        self._data.move_to_end(key)             # mark as most recently used
        return self._data[key]

    def put(self, key, value):
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)      # evict the least recently used (the front)

    def __len__(self):
        return len(self._data)


def expensive_square(k):
    return k * k

@lru_cache(maxsize=16)
def reference_square(k):
    return expensive_square(k)

ours = LRUCache(capacity=16)
def our_square(k):
    value = ours.get(k)
    if value is None:
        value = expensive_square(k)
        ours.put(k, value)
    return value

access_rng = random.Random(0)
requests = [min(int(access_rng.expovariate(1 / 12)), 60) for _ in range(5_000)]   # popular keys repeat often
for k in requests:
    assert our_square(k) == reference_square(k)

info = reference_square.cache_info()
assert (ours.hits, ours.misses, len(ours)) == (info.hits, info.misses, info.currsize)
print(f"✅ LRUCache matches functools.lru_cache: hits={ours.hits:,} misses={ours.misses:,} size={len(ours)} "
      f"→ hit rate {ours.hits / len(requests):.1%}")

✅ LRUCache matches functools.lru_cache: hits=2,849 misses=2,151 size=16 → hit rate 57.0%


## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Mutable default arguments

Default values are evaluated **once**, when the `def` line runs — not on every call.

In [54]:
def add_tag_wrong(tag, tags=[]):          # ❌ one list shared by every call
    tags.append(tag)
    return tags

print("❌", add_tag_wrong("cat"), add_tag_wrong("dog"), "| default is now:", add_tag_wrong.__defaults__)

def add_tag_right(tag, tags=None):        # ✅ None as a sentinel, new list per call
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

print("✅", add_tag_right("cat"), add_tag_right("dog"))

❌ ['cat', 'dog'] ['cat', 'dog'] | default is now: (['cat', 'dog'],)
✅ ['cat'] ['dog']


### ❌ Pitfall 2 — Changing a list while looping over it

In [55]:
numbers = [2, 4, 5, 6, 8]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)                  # ❌ shifts the items left, so the loop skips the next one
print("❌ remove evens while iterating:", numbers)

numbers = [2, 4, 5, 6, 8]
numbers = [n for n in numbers if n % 2 != 0]   # ✅ build a new list
print("✅ list comprehension          :", numbers)

❌ remove evens while iterating: [4, 5, 8]
✅ list comprehension          : [5]


### ❌ Pitfall 3 — Reusing an exhausted generator

In [56]:
readings = (float(v) for v in ["3.5", "7.25", "1.0"])
mean_value = sum(readings) / 3
try:
    print("❌ max of the same generator:", max(readings))
except ValueError as err:
    print(f"❌ second pass fails: ValueError: {err} (mean was {mean_value:.2f})")

readings = [float(v) for v in ["3.5", "7.25", "1.0"]]   # ✅ a list can be iterated many times
print(f"✅ list: mean {sum(readings) / len(readings):.2f}, max {max(readings)}")

❌ second pass fails: ValueError: max() iterable argument is empty (mean was 3.92)
✅ list: mean 3.92, max 7.25


### ❌ Pitfall 4 — Forgetting `await`

Without `await`, you get a coroutine *object*, which is truthy — so `if result:` silently passes.

In [57]:
async def is_toxic(text):
    await asyncio.sleep(0.01)
    return "hate" in text

verdict = is_toxic("have a nice day")              # ❌ missing await
print("❌ without await:", type(verdict).__name__, "| `if verdict:` treats it as", bool(verdict))
verdict.close()                                    # (tidy up the coroutine that never ran)

verdict = await is_toxic("have a nice day")        # ✅
print("✅ with await   :", verdict)

❌ without await: coroutine | `if verdict:` treats it as True
✅ with await   : False


### ❌ Pitfall 5 — A blocking call inside async code

`time.sleep`, `requests.get`, or a heavy CPU loop inside `async def` never yields to the event loop, so "concurrent" tasks run one after another.

In [58]:
async def blocking_task():
    time.sleep(0.2)                  # ❌ blocks the whole event loop
    return "done"

async def non_blocking_task():
    await asyncio.sleep(0.2)         # ✅ yields to the event loop
    return "done"

with Timer() as t_blocking:
    await asyncio.gather(*(blocking_task() for _ in range(5)))
with Timer() as t_non_blocking:
    await asyncio.gather(*(non_blocking_task() for _ in range(5)))
print(f"❌ time.sleep in async   : {t_blocking.seconds:.2f} s for 5 tasks (they ran one after another)")
print(f"✅ await asyncio.sleep   : {t_non_blocking.seconds:.2f} s for 5 tasks "
      f"→ {t_blocking.seconds / t_non_blocking.seconds:.1f}× faster")

❌ time.sleep in async   : 1.03 s for 5 tasks (they ran one after another)
✅ await asyncio.sleep   : 0.20 s for 5 tasks → 5.1× faster


### ❌ Pitfall 6 — A notebook-defined function in a process pool (spawn)

We run the mistake in a **separate Python process** so its crash doesn't clutter this notebook, and show the real error it prints.

In [59]:
broken_script = textwrap.dedent("""
    from concurrent.futures import ProcessPoolExecutor

    def double(x):          # defined in __main__, like a notebook cell
        return 2 * x

    if __name__ == "__main__":
        with ProcessPoolExecutor(max_workers=2) as pool:
            print(list(pool.map(double, [1, 2, 3])))
""")
# `python -c` has no importable __main__ file, just like a notebook kernel
proc = subprocess.run([sys.executable, "-c", broken_script], capture_output=True, text=True, timeout=120)
stderr_lines = proc.stderr.splitlines()
child_error = next((line for line in stderr_lines if line.startswith("AttributeError")), "(no AttributeError line found)")
parent_error = next((line for line in stderr_lines if "BrokenProcessPool" in line), "(no BrokenProcessPool line found)")
print(f"❌ exit code {proc.returncode}")
print(f"   each child process: {child_error}")
print(f"   the parent        : {parent_error}")

with ProcessPoolExecutor(max_workers=2) as pool:     # ✅ function imported from a real module file
    print("✅ importable worker:", list(pool.map(workers.square, [1, 2, 3])))

❌ exit code 1
   each child process: AttributeError: Can't get attribute 'double' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
   the parent        : concurrent.futures.process.BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.


✅ importable worker: [1, 4, 9]


## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct.

### 🟢 Exercise 1 — De-duplicate in O(n), keeping order
Return the items of `events` without duplicates, **keeping the first occurrence order**. No nested loops.

In [60]:
events = ["login", "click", "login", "scroll", "click", "logout"]
unique_events = None  # TODO
check("unique_events", unique_events, ["login", "click", "scroll", "logout"],
      hint="A set remembers what you've seen; dicts keep insertion order — dict.fromkeys(events) is a one-liner.")

⏳ unique_events: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
events = ["login", "click", "login", "scroll", "click", "logout"]
unique_events = list(dict.fromkeys(events))
check("unique_events", unique_events, ["login", "click", "scroll", "logout"])

# Explicit version with a set:
seen, ordered = set(), []
for e in events:
    if e not in seen:
        seen.add(e)
        ordered.append(e)
check("unique_events (set)", ordered, ["login", "click", "scroll", "logout"])
```
`list(set(events))` is O(n) too, but loses the order.
</details>

### 🟢 Exercise 2 — Fix the shared default
Write `make_batch(item, batch=None)` that appends `item` to `batch` and returns it, **without sharing a list between calls** when no batch is passed.

In [61]:
make_batch = None  # TODO: define a function (def make_batch(item, batch=None): ...)
check("make_batch", None if make_batch is None else (make_batch("a"), make_batch("b"), make_batch("c", ["x"])),
      (["a"], ["b"], ["x", "c"]), hint="Default to None, and create a new list inside the function when batch is None.")

⏳ make_batch: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def make_batch(item, batch=None):
    if batch is None:
        batch = []
    batch.append(item)
    return batch

check("make_batch", (make_batch("a"), make_batch("b"), make_batch("c", ["x"])), (["a"], ["b"], ["x", "c"]))
```
</details>

### 🟢 Exercise 3 — Keep only the last 3 events
Process the stream one event at a time and keep **only the 3 most recent** in `recent`, with O(1) work per event. Store the final contents as a list in `last_three`.

In [62]:
stream_of_events = ["e1", "e2", "e3", "e4", "e5"]
last_three = None  # TODO
check("last_three", last_three, ["e3", "e4", "e5"], hint="collections.deque(maxlen=3) drops the oldest item automatically on append.")

⏳ last_three: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
stream_of_events = ["e1", "e2", "e3", "e4", "e5"]
recent = deque(maxlen=3)
for event in stream_of_events:
    recent.append(event)
last_three = list(recent)
check("last_three", last_three, ["e3", "e4", "e5"])
```
A plain list with `pop(0)` also works but each `pop(0)` is O(n).
</details>

### 🟡 Exercise 4 — A thread-safe counter
`UnsafeCounter` loses increments under threads. Write `SafeCounter` with the same `increment()` method and `value` attribute that never loses an update. The checker runs 8 threads × 1,000 increments.

In [63]:
class UnsafeCounter:
    def __init__(self):
        self.value = 0

    def increment(self):
        current = self.value
        time.sleep(0)            # a thread switch can happen here
        self.value = current + 1

SafeCounter = None  # TODO: define class SafeCounter

def hammer(counter_cls, n_threads=8, times=1_000):
    counter_obj = counter_cls()
    def work():
        for _ in range(times):
            counter_obj.increment()
    threads = [threading.Thread(target=work) for _ in range(n_threads)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    return counter_obj.value

print("UnsafeCounter reached", hammer(UnsafeCounter), "of 8,000")
check("safe_counter", None if SafeCounter is None else hammer(SafeCounter), 8_000,
      hint="Create a threading.Lock() in __init__ and wrap the read-sleep-write in `with self._lock:`.")

UnsafeCounter reached 1039 of 8,000
⏳ safe_counter: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
class UnsafeCounter:
    def __init__(self):
        self.value = 0

    def increment(self):
        current = self.value
        time.sleep(0)
        self.value = current + 1


class SafeCounter:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            current = self.value
            time.sleep(0)
            self.value = current + 1


def hammer(counter_cls, n_threads=8, times=1_000):
    counter_obj = counter_cls()
    def work():
        for _ in range(times):
            counter_obj.increment()
    threads = [threading.Thread(target=work) for _ in range(n_threads)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    return counter_obj.value

check("safe_counter", hammer(SafeCounter), 8_000)
```
</details>

### 🟡 Exercise 5 — Count primes in parallel with processes
Count the primes below 200,000 by splitting `[0, 200_000)` into **4 equal chunks** and mapping `workers.count_primes_in_range` (it takes a `(start, stop)` tuple) over them with a `ProcessPoolExecutor`. Store the sum in `parallel_total`.

In [64]:
chunk_bounds = None      # TODO: [(0, 50_000), (50_000, 100_000), ...]
parallel_total = None    # TODO
check("parallel_total", parallel_total, 17_984,
      hint="bounds = [(i, i + 50_000) for i in range(0, 200_000, 50_000)]; then sum(pool.map(workers.count_primes_in_range, bounds)).")

⏳ parallel_total: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
chunk_bounds = [(start, start + 50_000) for start in range(0, 200_000, 50_000)]
with ProcessPoolExecutor(max_workers=4) as pool:
    parallel_total = sum(pool.map(workers.count_primes_in_range, chunk_bounds))
check("parallel_total", parallel_total, 17_984)
```
Note the chunks are not equally expensive: larger numbers need more trial divisions, so the last chunk is the slowest. Splitting work *evenly by cost* is a real concern in data-parallel pipelines.
</details>

### 🔴 Exercise 6 — Bounded async map with timeouts (interview classic)
Write `bounded_map(fn, items, limit, timeout)`: run the coroutine `fn(item)` for every item with **at most `limit` in flight**; any call slower than `timeout` seconds yields `None`. Return results **in input order**. (Think: "summarize 10,000 documents with an LLM API, max 20 concurrent, skip calls that hang.")

In [65]:
async def bounded_map(fn, items, limit, timeout):
    return None  # TODO


probe = {"in_flight": 0, "peak": 0}

async def sometimes_hangs(x):
    probe["in_flight"] += 1
    probe["peak"] = max(probe["peak"], probe["in_flight"])
    try:
        await asyncio.sleep(0.5 if x in (3, 7) else 0.05)
        return x
    finally:
        probe["in_flight"] -= 1

with Timer() as t:
    answers = await bounded_map(sometimes_hangs, range(10), limit=3, timeout=0.2)
check("bounded_map", None if answers is None else (list(answers), probe["peak"] <= 3, t.seconds < 1.0),
      ([0, 1, 2, None, 4, 5, 6, None, 8, 9], True, True),
      hint="asyncio.Semaphore(limit) + asyncio.wait_for(fn(item), timeout) inside try/except TimeoutError, then gather.")

⏳ bounded_map: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
async def bounded_map(fn, items, limit, timeout):
    semaphore = asyncio.Semaphore(limit)

    async def run_one(item):
        async with semaphore:
            try:
                return await asyncio.wait_for(fn(item), timeout)
            except TimeoutError:
                return None

    return await asyncio.gather(*(run_one(item) for item in items))


probe = {"in_flight": 0, "peak": 0}

async def sometimes_hangs(x):
    probe["in_flight"] += 1
    probe["peak"] = max(probe["peak"], probe["in_flight"])
    try:
        await asyncio.sleep(0.5 if x in (3, 7) else 0.05)
        return x
    finally:
        probe["in_flight"] -= 1

with Timer() as t:
    answers = await bounded_map(sometimes_hangs, range(10), limit=3, timeout=0.2)
check("bounded_map", (list(answers), probe["peak"] <= 3, t.seconds < 1.0), ([0, 1, 2, None, 4, 5, 6, None, 8, 9], True, True))
```

Follow-ups interviewers ask: *put the timeout inside the semaphore* (so waiting for a permit doesn't count as hanging), *add retries with backoff*, and *stream results as they finish* with `asyncio.as_completed`.
</details>

## 🚀 Mini Project: Concurrent Dataset Downloader

**Goal:** download and summarize 16 real CSV datasets from the public [seaborn-data repository](https://github.com/mwaskom/seaborn-data) three ways — **sequentially**, with a **thread pool**, and with **asyncio** — measure each fairly, verify they produce identical results, and let the measurements decide which approach wins.

**Steps:** pick the files → write the processing function → implement three downloaders → benchmark with a warm-up and repeated rounds → check the CPU-bound processing step separately → draw conclusions from the numbers.

### Step 1 — The files and a network check

In [66]:
DATASET_NAMES = ["tips", "iris", "penguins", "flights", "titanic", "mpg", "exercise", "attention",
                 "anscombe", "car_crashes", "dots", "fmri", "geyser", "glue", "healthexp", "planets"]
DATASET_URLS = [SEABORN_DATA.format(name) for name in DATASET_NAMES]

try:
    httpx.get(DATASET_URLS[0], timeout=15).raise_for_status()
    NETWORK_OK = True
    print(f"✅ network OK — {len(DATASET_URLS)} files to download")
except httpx.HTTPError as err:
    NETWORK_OK = False
    print(f"⏭️ Skipped: cannot reach raw.githubusercontent.com ({err!r}). This project needs internet access.")

✅ network OK — 16 files to download


### Step 2 — The processing step

For each file we compute the number of rows and columns, how many columns are fully numeric, and a SHA-256 fingerprint (to prove every method downloaded identical bytes). The function lives in an importable module so we can also try it in a process pool in Step 5.

In [67]:
PROJECT_SOURCE = '''"""Mini-project workers: summarize a downloaded CSV file."""
import csv
import hashlib
import io


def _is_number(text):
    try:
        float(text)
        return True
    except ValueError:
        return False


def summarize_csv(name, raw_bytes):
    rows = list(csv.reader(io.StringIO(raw_bytes.decode("utf-8"))))
    header, body = rows[0], [r for r in rows[1:] if r]
    numeric_columns = sum(
        all(_is_number(r[i]) for r in body if i < len(r) and r[i] != "") for i in range(len(header))
    )
    return {
        "name": name,
        "rows": len(body),
        "columns": len(header),
        "numeric_columns": numeric_columns,
        "sha256": hashlib.sha256(raw_bytes).hexdigest()[:12],
    }
'''
(OUTPUT_DIR / "downloader_workers.py").write_text(PROJECT_SOURCE)
importlib.invalidate_caches()
import downloader_workers
downloader_workers = importlib.reload(downloader_workers)

print(downloader_workers.summarize_csv("demo", b"a,b,label\n1,2.5,cat\n3,,dog\n"))

{'name': 'demo', 'rows': 2, 'columns': 3, 'numeric_columns': 2, 'sha256': '9eaa7ed69651'}


### Step 3 — Three downloaders

All three reuse HTTP connections (a *client* keeps connections open), because opening a new TLS connection per file would dominate the timing.
- **Sequential:** one `httpx.Client`, one file after another.
- **Threads:** a `ThreadPoolExecutor`; each worker thread gets its *own* client through `threading.local()`, a safe default when you're unsure a client object is thread-safe.
- **asyncio:** one `httpx.AsyncClient` with a `Semaphore` to cap concurrency.

In [68]:
MAX_CONCURRENCY = 8


def download_sequential(urls):
    with httpx.Client(timeout=30) as client:
        return [client.get(u).raise_for_status().content for u in urls]


def download_threads(urls, max_workers=MAX_CONCURRENCY):
    local = threading.local()
    clients = []

    def get(url):
        if not hasattr(local, "client"):
            local.client = httpx.Client(timeout=30)
            clients.append(local.client)
        return local.client.get(url).raise_for_status().content

    try:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            return list(pool.map(get, urls))
    finally:
        for c in clients:
            c.close()


async def download_async(urls, max_concurrency=MAX_CONCURRENCY):
    semaphore = asyncio.Semaphore(max_concurrency)
    async with httpx.AsyncClient(timeout=30) as client:
        async def get(url):
            async with semaphore:
                response = await client.get(url)
                return response.raise_for_status().content
        return await asyncio.gather(*(get(u) for u in urls))


if NETWORK_OK:
    sample = download_sequential(DATASET_URLS[:2])
    print("smoke test:", [downloader_workers.summarize_csv(n, b) for n, b in zip(DATASET_NAMES[:2], sample)])
else:
    print("⏭️ Skipped (no network).")

smoke test:

 [{'name': 'tips', 'rows': 244, 'columns': 7, 'numeric_columns': 3, 'sha256': 'e54cc4d2ce1b'}, {'name': 'iris', 'rows': 150, 'columns': 5, 'numeric_columns': 4, 'sha256': '9cc1c345c71b'}]


### Step 4 — Benchmark fairly

- **Warm-up round** first (untimed), so no method benefits from caches the others warmed.
- **3 rounds per method**, interleaved, and we report the **median** to reduce noise from the network.
- Every round must produce **identical summaries**, or the speed doesn't count.

In [69]:
ROUNDS = 3
timings = {"sequential": [], "threads": [], "asyncio": []}
reference_summaries = None

if NETWORK_OK:
    download_threads(DATASET_URLS)                                    # warm-up (untimed)
    for round_no in range(ROUNDS):
        for method in timings:
            with Timer() as t:
                if method == "sequential":
                    payloads = download_sequential(DATASET_URLS)
                elif method == "threads":
                    payloads = download_threads(DATASET_URLS)
                else:
                    payloads = await download_async(DATASET_URLS)
            timings[method].append(t.seconds)
            summaries = [downloader_workers.summarize_csv(n, b) for n, b in zip(DATASET_NAMES, payloads)]
            if reference_summaries is None:
                reference_summaries = summaries
            assert summaries == reference_summaries, f"{method} returned different data!"
    total_bytes = sum(len(b) for b in payloads)
    print(f"✅ all {ROUNDS * len(timings)} runs returned identical data ({total_bytes / 1e3:,.0f} KB per run)\n")
    print(f"{'file':<12} {'rows':>6} {'cols':>5} {'numeric':>8}  sha256")
    for s in reference_summaries:
        print(f"{s['name']:<12} {s['rows']:>6,} {s['columns']:>5} {s['numeric_columns']:>8}  {s['sha256']}")
else:
    print("⏭️ Skipped (no network).")

✅ all 9 runs returned identical data (229 KB per run)

file           rows  cols  numeric  sha256
tips            244     7        3  e54cc4d2ce1b
iris            150     5        4  9cc1c345c71b
penguins        344     7        4  e07636bd8af7
flights         144     3        2  237d834127d9
titanic         891    15        6  81787d320d7f
mpg             398     9        7  c14b8b855ea7
exercise         90     6        3  d67ff5896d7d
attention        60     5        4  5c1de4b2a7cb
anscombe         44     3        2  a0c1f636aa03
car_crashes      51     8        7  78ac44c0f6d4
dots            848     5        3  dd8ed5e18358
fmri          1,064     5        2  8a0bfdce94da
geyser          272     3        2  ce8f6bd15967
glue             64     5        2  0c6b840da0a2
healthexp       274     4        3  ba4178979b7b
planets       1,035     6        5  a6d10044887e


In [70]:
if NETWORK_OK:
    medians = {m: statistics.median(v) for m, v in timings.items()}
    print(f"{'method':<11} {'median':>8} {'all rounds':>26} {'speedup':>8}")
    for m, med in medians.items():
        rounds_text = ", ".join(f"{v:.2f}" for v in timings[m])
        print(f"{m:<11} {med:>7.2f}s {rounds_text:>26} {medians['sequential'] / med:>7.1f}×")
else:
    print("⏭️ Skipped (no network).")

method        median                 all rounds  speedup
sequential     0.23s           0.23, 0.25, 0.18     1.0×
threads        0.09s           0.09, 0.11, 0.07     2.7×
asyncio        0.08s           0.07, 0.09, 0.08     2.8×


### Step 5 — Is the *processing* step worth parallelizing?

Parsing is CPU work, so threads won't help — but are processes worth their overhead for 16 small files? Measure it.

In [71]:
if NETWORK_OK:
    with Timer() as t_parse_seq:
        parsed_seq = [downloader_workers.summarize_csv(n, b) for n, b in zip(DATASET_NAMES, payloads)]
    with Timer() as t_parse_proc:
        with ProcessPoolExecutor(max_workers=4) as pool:
            parsed_proc = list(pool.map(downloader_workers.summarize_csv, DATASET_NAMES, payloads))
    assert parsed_seq == parsed_proc == reference_summaries
    print(f"summarize 16 files → sequential {t_parse_seq.seconds * 1000:.1f} ms | "
          f"process pool {t_parse_proc.seconds * 1000:.0f} ms (including start-up)")
else:
    print("⏭️ Skipped (no network).")

summarize 16 files → sequential 5.2 ms | process pool 51 ms (including start-up)


### Step 6 — Conclusions computed from the measurements

In [72]:
if NETWORK_OK:
    fastest = min(medians, key=medians.get)
    speedup = medians["sequential"] / medians[fastest]
    per_file_ms = medians["sequential"] / len(DATASET_URLS) * 1000
    download_share = medians["sequential"] / (medians["sequential"] + t_parse_seq.seconds)
    print(f"1. Fastest downloader: {fastest} — {speedup:.1f}× faster than sequential "
          f"(median {medians[fastest]:.2f} s vs {medians['sequential']:.2f} s).")
    gap = abs(medians["threads"] - medians["asyncio"]) / min(medians["threads"], medians["asyncio"])
    if gap < 0.5:
        print(f"2. Threads and asyncio were within {gap:.0%} of each other: for {len(DATASET_URLS)} requests both remove the waiting.")
    else:
        slower = max(("threads", "asyncio"), key=medians.get)
        print(f"2. {slower} was {gap:.0%} slower than the other method — overhead matters at this small scale.")
    print(f"3. Sequential spent ≈{per_file_ms:.0f} ms per file, and downloading was {download_share:.0%} of the "
          f"sequential pipeline's time → the job is I/O-bound.")
    if t_parse_proc.seconds > t_parse_seq.seconds:
        print(f"4. For processing, the process pool was {t_parse_proc.seconds / t_parse_seq.seconds:,.0f}× SLOWER than a "
              f"plain loop: start-up and pickling cost more than {t_parse_seq.seconds * 1000:.1f} ms of parsing. Keep it sequential.")
    else:
        print(f"4. The process pool sped up processing {t_parse_seq.seconds / t_parse_proc.seconds:.1f}× — worth it at this size.")
else:
    print("⏭️ Skipped (no network).")

1. Fastest downloader: asyncio — 2.8× faster than sequential (median 0.08 s vs 0.23 s).
2. Threads and asyncio were within 1% of each other: for 16 requests both remove the waiting.
3. Sequential spent ≈15 ms per file, and downloading was 98% of the sequential pipeline's time → the job is I/O-bound.
4. For processing, the process pool was 10× SLOWER than a plain loop: start-up and pickling cost more than 5.2 ms of parsing. Keep it sequential.


**Stretch goals**
1. Add **retries with exponential backoff** for timeouts and 5xx responses, and test it by including one URL that returns 404.
2. Try `MAX_CONCURRENCY` values of 1, 2, 4, 8, 16 and plot median time vs concurrency. Where does it stop improving, and why?
3. **Stream** a large file to disk in chunks with `client.stream("GET", url)` so memory stays flat, and show progress with `asyncio.as_completed`.
4. Make processing expensive (e.g. parse `diamonds.csv`, ~54,000 rows, and compute per-column statistics in pure Python) and find the file size at which the process pool starts to win.

### 🗣️ How to talk about this in an interview
- "I benchmarked sequential, thread-pool, and asyncio downloaders on 16 real files with a warm-up round, interleaved repeats, and medians, and asserted all methods returned byte-identical data before comparing speed."
- "Downloading was I/O-bound, so both threads and asyncio removed the waiting and gave a large speedup; I capped concurrency with a semaphore to be polite to the server."
- "Parsing is CPU-bound, but a process pool was slower for small files because start-up and pickling dominate — so I measured instead of assuming 'parallel is faster'."
- "On macOS, process pools use spawn, so worker functions live in an importable module — the same reason PyTorch `Dataset` classes must be importable when `num_workers > 0`."
- "In production I'd add retries with backoff and jitter, per-host limits, timeouts, and streaming for big files."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What is the difference between `is` and `==`?**

<details><summary>Show answer</summary>

- **30-second answer:** `==` compares values by calling `__eq__`; `is` checks identity — whether both names refer to the same object. Use `is` for singletons like `None`, `==` for everything else.
- **Go deeper:** CPython caches small ints (-5 to 256) and interns some strings, so `a is b` can be `True` for equal small values by accident — never rely on it. A class without `__eq__` falls back to identity for `==` too.
- **❌ Common wrong answer:** "`is` is a faster `==`."

</details>

**Q2. How does Python manage memory?**

<details><summary>Show answer</summary>

- **30-second answer:** CPython uses reference counting — each object tracks how many references point to it and is freed as soon as the count hits zero — plus a generational cyclic garbage collector that finds unreachable reference cycles.
- **Go deeper:** `sys.getrefcount` shows counts; `weakref` lets you observe without keeping objects alive; `gc.collect()` forces a cycle collection. Small objects come from the `pymalloc` allocator, which may keep freed memory for reuse rather than return it to the OS. Since 3.12, objects like `None` are "immortal" (PEP 683).
- **❌ Common wrong answer:** "Python only has a garbage collector that runs periodically" (it misses reference counting, which frees most objects immediately).

</details>

**Q3. What is the GIL, why does it exist, and when does it matter?**

<details><summary>Show answer</summary>

- **30-second answer:** The Global Interpreter Lock lets only one thread execute Python bytecode at a time in a CPython process. It keeps reference counting and interpreter internals safe and simple. It makes CPU-bound pure-Python threads no faster, but I/O-bound threads work well because blocking I/O releases the GIL.
- **Go deeper:** C extensions (NumPy, PyTorch, hashlib) release the GIL in heavy operations, so threads can use multiple cores there. PEP 703 introduced a free-threaded build: experimental in 3.13, officially supported but still optional in 3.14. The GIL does not make your own compound operations atomic.
- **❌ Common wrong answer:** "Because of the GIL, Python threads are useless" or "the GIL makes my code thread-safe."

</details>

**Q4. Threads, processes, or asyncio — how do you choose?**

<details><summary>Show answer</summary>

- **30-second answer:** CPU-bound pure Python → processes. I/O-bound with a handful to hundreds of tasks, or blocking libraries → threads. I/O-bound with very many connections and async libraries available → asyncio. Always measure.
- **Go deeper:** Processes pay start-up and pickling costs and don't share memory; threads share memory so need locks or queues; asyncio switches only at `await`, so one blocking call stalls everything (`asyncio.to_thread` is the escape hatch). Real systems mix them: an async API server that sends CPU work to a process pool.
- **❌ Common wrong answer:** "asyncio makes CPU-heavy code faster."

</details>

**Q5. Why can't a list be a dictionary key? What makes an object hashable?**

<details><summary>Show answer</summary>

- **30-second answer:** Dicts locate keys by their hash. A list is mutable, so its hash would change after insertion and the key could never be found again; Python therefore makes lists unhashable. Hashable objects implement `__hash__` consistent with `__eq__` and don't change in ways that affect either.
- **Go deeper:** Tuples are hashable only if their items are. `@dataclass(frozen=True)` generates both methods. Defining `__eq__` without `__hash__` sets `__hash__` to `None`. Average lookup is O(1); many collisions degrade it to O(n).
- **❌ Common wrong answer:** "Lists can't be keys because they're not strings or numbers."

</details>

**Q6. What is a closure, and why does `[lambda: i for i in range(3)]` return 2, 2, 2?**

<details><summary>Show answer</summary>

- **30-second answer:** A closure is a function that keeps access to variables from the scope where it was defined. It captures the *variable* (a cell), not its value, so all three lambdas read `i` when called — after the loop ended with `i = 2`. Fix with `lambda i=i: i` or `functools.partial`.
- **Go deeper:** Inspect captured cells with `f.__closure__` and `f.__code__.co_freevars`. `nonlocal` lets the inner function rebind the captured variable. Decorators are closures.
- **❌ Common wrong answer:** "Lambdas copy the loop variable's value when they are created."

</details>

**Q7. When would you use a generator instead of a list?**

<details><summary>Show answer</summary>

- **30-second answer:** When data is large, unbounded, or streamed and you only need one pass: a generator produces items lazily with O(1) memory. Use a list when you need `len`, indexing, or several passes.
- **Go deeper:** Generators pause at `yield`, keeping local state; they can be consumed only once. Data loaders, file readers, and token streaming from LLM APIs are all generator-shaped. `itertools` builds lazy pipelines.
- **❌ Common wrong answer:** "Generators are always faster than lists." (They save memory; raw speed is often similar or slightly slower.)

</details>

### 💻 Coding

**Q8. Implement an LRU cache with O(1) `get` and `put`.**

<details><summary>Show answer</summary>

- **30-second answer:** Use an `OrderedDict`: on `get`, `move_to_end(key)`; on `put`, set the value, `move_to_end`, and if over capacity `popitem(last=False)` to evict the oldest.
- **Go deeper:** Without `OrderedDict`, combine a dict (key → node) with a doubly linked list for O(1) moves and evictions. For function results, `functools.lru_cache(maxsize=...)` does this and reports `cache_info()`. Discuss thread safety (add a lock) and TTL expiry if asked.
- **❌ Common wrong answer:** Using a list to track order — `list.remove(key)` is O(n).

</details>

**Q9. You need to send 10,000 prompts to an LLM API that allows 50 concurrent requests. Sketch the code.**

<details><summary>Show answer</summary>

- **30-second answer:** Use the provider's async client, an `asyncio.Semaphore(50)`, and `asyncio.gather` (or `as_completed` to save results as they arrive). Wrap each call with retries using exponential backoff plus jitter for 429/5xx, and a timeout.
- **Go deeper:** Rate limits are usually requests *and* tokens per minute, so add a token-bucket limiter; honour `Retry-After`; write results incrementally so a crash doesn't lose work; consider the provider's batch API for large offline jobs (cheaper, not real-time).
- **❌ Common wrong answer:** `asyncio.gather` on all 10,000 calls with no limit, or a `for` loop with `requests.post` inside `async def`.

</details>

**Q10. Implement a producer–consumer pipeline with threads.**

<details><summary>Show answer</summary>

- **30-second answer:** Share a `queue.Queue(maxsize=n)` between producer and consumer threads: producers `put`, consumers loop on `get`, and a sentinel (e.g. `None`) per consumer signals shutdown; `join` the threads at the end.
- **Go deeper:** A bounded queue gives *backpressure* — fast producers block instead of filling memory. `queue.Queue` handles locking internally; use `task_done()`/`join()` to wait for all items. The async equivalent is `asyncio.Queue`.
- **❌ Common wrong answer:** Sharing a plain list with busy-waiting loops like `while not items: pass`.

</details>

### 🐛 Debugging Scenarios

**Q11. A function `def add(item, items=[])` returns items from previous calls. What's happening?**

<details><summary>Show answer</summary>

- **30-second answer:** Default values are evaluated once, when the function is defined, so every call without `items` shares the same list. Use `items=None` and create a new list inside.
- **Go deeper:** You can see the shared object in `add.__defaults__`. Linters flag this (Ruff/flake8-bugbear rule B006). The same applies to dict and set defaults.
- **❌ Common wrong answer:** "Python caches function return values."

</details>

**Q12. Your team moved CPU-heavy text cleaning to a `ThreadPoolExecutor` with 16 threads and it got no faster. Why, and what do you do?**

<details><summary>Show answer</summary>

- **30-second answer:** The cleaning is pure-Python CPU work, so the GIL lets only one thread run at a time. Switch to `ProcessPoolExecutor` (or `datasets.map(num_proc=...)`), with batched inputs via `chunksize` to cut pickling overhead.
- **Go deeper:** Profile first to confirm it's CPU-bound. Alternatives: vectorize with pandas/NumPy, use a Rust-backed library (e.g. Hugging Face `tokenizers`), or try a free-threaded Python build if all dependencies support it.
- **❌ Common wrong answer:** "Add more threads."

</details>

**Q13. In a Jupyter notebook on macOS, `ProcessPoolExecutor` fails with `BrokenProcessPool` and "Can't get attribute 'my_func' on `<module '__main__'>`". Why?**

<details><summary>Show answer</summary>

- **30-second answer:** macOS uses the `spawn` start method: each worker starts a fresh interpreter and imports the function by module and name. A function defined in a notebook cell lives in the kernel's `__main__`, which the worker can't import. Move the function to a `.py` module and import it.
- **Go deeper:** Lambdas and nested functions can't be pickled at all. In scripts, protect pool creation with `if __name__ == "__main__":`. PyTorch `DataLoader(num_workers>0)` has the same requirement for `Dataset` classes.
- **❌ Common wrong answer:** "Multiprocessing doesn't work on macOS."

</details>

**Q14. An async FastAPI endpoint handles one request fine but latency explodes under load. What's the likely cause?**

<details><summary>Show answer</summary>

- **30-second answer:** Something blocking runs inside `async def` — `requests.get`, `time.sleep`, a synchronous database driver, or CPU-heavy model inference — which freezes the single event loop so requests queue up behind it.
- **Go deeper:** Use async libraries (`httpx.AsyncClient`, async DB drivers), wrap unavoidable blocking calls with `asyncio.to_thread` / `run_in_executor`, define the endpoint with plain `def` so FastAPI runs it in a thread pool, and move heavy inference to a separate worker or inference server. asyncio's debug mode logs slow callbacks.
- **❌ Common wrong answer:** "Add more `await` keywords" (awaiting nothing async doesn't help) or "async is just slow."

</details>

### 🏗️ Design

**Q15. Design the data pipeline to embed 5 million documents: download from S3, clean text, call a hosted embedding API, write to a vector database. Where do threads, processes, and asyncio go?**

<details><summary>Show answer</summary>

- **30-second answer:** Downloads: async or threads (I/O). Cleaning: a process pool or `datasets.map(num_proc)` (CPU). Embedding API: asyncio with a semaphore/token bucket sized to the rate limit, batching many texts per request, retries with backoff. Vector DB writes: async batched upserts. Connect stages with bounded queues for backpressure.
- **Go deeper:** Make it resumable (checkpoint processed IDs, idempotent upserts), deduplicate before paying for embeddings, track tokens/minute and cost, and log failures to a dead-letter queue. At bigger scale, split stages into separate services or use Ray/Spark.
- **❌ Common wrong answer:** "Run everything in one big multithreaded loop" or "use as many processes as possible for the API calls."

</details>

## 🧪 Quick Quiz

Predict the output, then reveal.

**1.** `fs = [lambda: i for i in range(3)]; print([f() for f in fs])`
<details><summary>Answer</summary>

`[2, 2, 2]` — every lambda reads the same variable `i` after the loop has finished.
</details>

**2.** `t = ([],)` then `t[0] += [1]`. What happens, and what is `t` afterwards?
<details><summary>Answer</summary>

It raises `TypeError: 'tuple' object does not support item assignment` — **and** `t` is `([1],)`. `+=` first extends the list in place (succeeds), then tries to store the result back into the tuple slot (fails).
</details>

**3.** `grid = [[0] * 2] * 2; grid[0][0] = 1; print(grid)`
<details><summary>Answer</summary>

`[[1, 0], [1, 0]]` — both rows are the same list object.
</details>

**4.** `def f(x, acc=[]): acc.append(x); return acc` — what does `f(1); print(f(2))` print?
<details><summary>Answer</summary>

`[1, 2]` — the default list is created once and shared between calls.
</details>

**5.** Two coroutines each run `time.sleep(0.5)` (not `asyncio.sleep`). About how long does `await asyncio.gather(a(), b())` take? (a) 0.5 s (b) 1.0 s (c) 0 s
<details><summary>Answer</summary>

**(b) about 1.0 s** — `time.sleep` blocks the event loop, so the coroutines run one after the other.
</details>

## 📚 Resources

### 📖 Official Docs
- [Data model](https://docs.python.org/3/reference/datamodel.html) — objects, identity, `__hash__`/`__eq__`, and every special method
- [Programming FAQ: lambdas in a loop](https://docs.python.org/3/faq/programming.html#why-do-lambdas-defined-in-a-loop-with-different-values-all-return-the-same-result) — the official explanation of late binding
- [gc — Garbage Collector interface](https://docs.python.org/3/library/gc.html) · [dis — Disassembler for Python bytecode](https://docs.python.org/3/library/dis.html)
- [concurrent.futures — Launching parallel tasks](https://docs.python.org/3/library/concurrent.futures.html) — thread and process pools
- [multiprocessing: contexts and start methods](https://docs.python.org/3/library/multiprocessing.html#contexts-and-start-methods) — spawn, fork, forkserver
- [Coroutines and tasks (asyncio)](https://docs.python.org/3/library/asyncio-task.html) — `gather`, `TaskGroup`, `timeout`, `to_thread`
- [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html) — running without the GIL
- [TimeComplexity (Python wiki)](https://wiki.python.org/moin/TimeComplexity) — Big-O of list, dict, set, deque operations

### 🎥 Videos
- [Nina Zakharenko — Memory Management in Python: The Basics (PyCon 2016)](https://www.youtube.com/watch?v=F6u5rhUQ6dU) (27 min) — names, references, reference counting and the garbage collector with clear diagrams; a perfect companion to Sections 2–3
- [David Beazley — Understanding the Python GIL](https://www.youtube.com/watch?v=Obt-vMVdM8s) (46 min) — the classic deep-dive into how the GIL schedules threads and why CPU-bound threads can even get slower
- [Łukasz Langa — import asyncio: Learn Python's AsyncIO #1: The Async Ecosystem](https://www.youtube.com/watch?v=Xbl7XjFYsN4) (50 min) — a CPython core developer explains event loops and coroutines from first principles

### 📄 Papers & Design Documents
- [PEP 703 – Making the Global Interpreter Lock Optional in CPython](https://peps.python.org/pep-0703/) — why and how the GIL is being made optional
- [PEP 683 – Immortal Objects, Using a Fixed Refcount](https://peps.python.org/pep-0683/) — why `sys.getrefcount(None)` is a huge constant

### 📘 Books & Courses
- [Real Python — Speed Up Your Python Program With Concurrency](https://realpython.com/python-concurrency/) — threads, asyncio, and multiprocessing side by side
- [Python behind the scenes (Ten Thousand Meters blog)](https://tenthousandmeters.com/tag/python-behind-the-scenes/) — a free series on how CPython compiles and runs code, including the GIL and async/await
- [CPython internal docs: garbage collector design](https://github.com/python/cpython/blob/main/InternalDocs/garbage_collector.md) — the real implementation notes, for the curious

### 🏋️ Practice
- [HackerRank — Python practice](https://www.hackerrank.com/domains/python) — short problems to drill built-ins, closures, and data structures

## 📝 Summary Cheat Sheet

| Concept | What it does | Key API / rule |
|---|---|---|
| Identity vs equality | same object vs same value | `is` (only for `None`), `==`, `id()` |
| Names & refcounts | variables are name tags; freed at refcount 0 | `sys.getrefcount`, `weakref.ref`, `del` removes a name |
| Garbage collection | frees unreachable reference cycles | `gc.collect()`, `gc.get_threshold()` |
| Copies | shallow shares nested objects; deep copies all | `copy.copy`, `copy.deepcopy`, `[[0]*n for _ in range(n)]` |
| Scope | Local → Enclosing → Global → Built-in | `global`, `nonlocal`, `UnboundLocalError` |
| Closures | functions remember variables (late binding) | `lambda i=i: ...`, `functools.partial` |
| Hashing | O(1) dict/set lookup | `__hash__` + `__eq__`, tuples & `@dataclass(frozen=True)` as keys |
| Complexity | pick structures by cost | `in set` O(1), `in list` O(n), `deque` for both ends |
| Generators | lazy, one pass, O(1) memory | `yield`, `(x for x in ...)`, `itertools.batched` |
| Bytecode | what the VM runs | `dis.dis(fn)`; `x += 1` is not atomic |
| GIL | one thread runs bytecode at a time | threads for I/O; `threading.Lock` for shared state |
| Processes | parallel CPU work, own GIL each | `ProcessPoolExecutor`, `chunksize`, importable workers (spawn) |
| asyncio | many I/O tasks on one thread | `async/await`, `gather`, `Semaphore`, `timeout`, `to_thread`; top-level `await` in Jupyter |
| AI patterns | right tool per workload | API calls → async + semaphore + backoff; preprocessing → processes; tensors → vectorize |

## ➡️ What's Next

**[01 · NumPy](../01_Core_Scientific_Computing/01_NumPy.ipynb)** — you now know why pure-Python loops are slow (bytecode, boxed objects, the GIL); NumPy's answer is to push the loop into compiled code over contiguous arrays, which is the foundation of every data and deep-learning library that follows.